# Portfolio edition
This is a source-only publishing copy of the original coursework notebook. Saved outputs and account-specific notebook metadata were cleared. Run the notebook with the documented data and compute requirements; historical results are not represented as a fresh execution.


**32513 ADVANCED DATA ANALYTICS · ASSIGNMENT 2**

# Does Marginal Likelihood Select the Best Forecast?

*A from-scratch exact Gaussian-process study of model selection, joint forecast density and uncertainty calibration*

---

| Project information | Value |
|---|---|
| **Project type** | Option 1 — study and implement a fundamental ML model |
| **Model** | Exact locally periodic Gaussian process |
| **Learning criteria** | Log marginal likelihood (LML) vs blocked rolling-origin cross-validation (CV) |
| **Deployment task** | Joint probabilistic forecast of the next 11 annual observations |
| **Primary metric** | Joint negative log predictive density (JNLPD; lower is better) |

This notebook asks when a GP hyperparameter criterion based on the observed training sample agrees—or disagrees—with a criterion aligned to future block forecasting. It implements the covariance, posterior, marginal likelihood, analytic log-parameter gradients, blocked-CV objective and joint predictive score explicitly with NumPy/SciPy. Scikit-learn is used only as an independent correctness reference in E1. The study proceeds from verified implementation to controlled specification, controlled misspecification, real sunspot forecasts, calibration failures and numerical limits.

> **Reading guide.** Section 0 records the two dates that matter for research integrity. Sections 1–7 define and verify the method. Sections 8–13 contain the frozen experiments. Sections 14–16 synthesize the evidence, document the implementation process and list references.

## 0. Analysis protocol and research-integrity record

**Research question.** For a locally periodic Gaussian process used for 11-year block forecasting, how do hyperparameters selected by maximising log marginal likelihood and by minimising blocked rolling-origin joint predictive loss compare in out-of-sample probabilistic accuracy and calibration, and how does this comparison change under model misspecification?

**Dates retained for research integrity.** The analysis protocol was frozen on **20 September 2026**, before scientific E2/E3 outcomes were generated. On **21 September 2026**, also before those outcomes, the qualitative H1 expectation was converted into the directional mean-$\Delta$ rule below. Ordinary result sections do not repeat execution dates; technical events are ordered by stage in the implementation log.

**Frozen comparison.** One model family; two learning criteria. Both methods use identical parameter bounds, eight saved starts, data, outer blocks and numerical safeguards. LML uses analytic gradients. CV uses SciPy L-BFGS-B with `jac='3-point'` and `finite_diff_rel_step=1e-5`.

Let

$$
\Delta_{\mathrm{JNLPD}}
=\mathrm{JNLPD}_{\mathrm{CV}}-\mathrm{JNLPD}_{\mathrm{LML}},
$$

so $\Delta<0$ favours CV and $\Delta>0$ favours LML.

**Frozen hypotheses.** H1: under correctly specified synthetic data, $\mathbb{E}[\Delta]\geq0$; a CV advantage is claimed only if the paired-bootstrap 95% interval lies wholly below zero. H2: under gradual period drift, $\mathbb{E}[\Delta]<0$; support requires that interval to lie wholly below zero. H3: on sunspots, coverage may deteriorate at longer horizons and in high lag-11 activity-proxy groups; negative lower bounds are a separate Gaussian-support diagnostic. Such patterns would be consistent with—not proof of—constant-noise or support limitations.

**Frozen scope.** Exact locally periodic GP; zero mean after training-only standardisation; annual statsmodels sunspots only for real data; ten non-overlapping 11-year outer blocks covering 1899–2008; four expanding 11-year inner validation folds; 50 correctly specified and 50 period-drift synthetic replicates; probabilistic lag-11 seasonal-naive baseline. Excluded: extra datasets or kernels, ARIMA, neural networks, sparse or heteroscedastic GPs, target transformations, conformal prediction, MCMC and significance tests on the ten real blocks.

**Pre-freeze disclosure.** Before the freeze, the sunspot series was inspected for suitability, including a heuristic full-series cycle-length check (approximately 8–15 years). Two exploratory optimiser experiments used historical windows ending in 1950 and 1948; the second included a finite-difference step-size check. Both produced fitted periods around 10.5–10.8 years and did not alter the bounds. No outer-test forecast block was scored or compared. The period bound $p\in[8,14]$ is a plausibility range based on published minimum-to-minimum cycle durations (WDC-SILSO; NASA/TM-20220002905, §8.3.3; ESA). No outer-test outcome influenced the hypotheses, model, bounds, splits or evaluation protocol.

## 1. Environment, persistent storage and recovery

Run this notebook from top to bottom on a CPU runtime. In the first code cell, authorize Google Drive mounting. Every completed seed and outer block is saved to **My Drive / A2_GP_LML_vs_CV / frozen_rerun_01**. The cell verifies storage access before computation begins. No experiment silently falls back to temporary Colab storage.

If Colab disconnects, reconnect, run all cells and authorize Drive again using the same RUN_ID. Completed, validated records are loaded; only unfinished work is recomputed. Each seed prints a start message and a saved confirmation. The currently running seed can take several minutes without a new message. Do not run two copies against the same RUN_ID simultaneously.

If the environment changes, the notebook stops rather than combining results from different package versions. Restore the versions recorded in run_manifest.json, or choose a new RUN_ID for a separate run. Never change the frozen scientific parameters to repair a connection problem.

The final cell validates record counts, writes a checksum manifest and creates a ZIP in the same Drive directory. Notebook outputs are saved separately by Colab: save and download the executed .ipynb as well. The ZIP preserves numerical evidence even if browser output rendering fails; it is not a substitute for the executed notebook.

Drive reduces the risk of losing completed work but cannot guarantee service availability or preservation of the seed currently in memory. If a write fails, execution stops visibly. Resume after restoring access.

This revision clears previous outputs for an honest fresh run. Prior local and Colab executions are retained in the local review evidence archive. The scientific protocol remains frozen; persistence, progress and reporting were revised after the first Colab run lost its temporary files.

In [ ]:
import sys, platform, warnings, subprocess, importlib.util, json
from pathlib import Path

LOCAL_PACKAGES = Path.cwd() / '.python_packages'
if LOCAL_PACKAGES.exists():
    sys.path.insert(0, str(LOCAL_PACKAGES))

REQUIRED_PACKAGES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'sklearn': 'scikit-learn', 'statsmodels': 'statsmodels', 'matplotlib': 'matplotlib'
}
missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])

import numpy as np
import pandas as pd
import scipy
from scipy.linalg import cho_factor, cho_solve, cholesky, solve_triangular
from scipy.optimize import minimize
from scipy.stats import multivariate_normal, norm
import sklearn
import statsmodels.api as sm
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260920
E1_SEED = 20260921
BOOTSTRAP_SEED = 20260922
BOOTSTRAP_RESAMPLES = 10_000
SCIENTIFIC_SEEDS = tuple(range(50))
ENGINEERING_SEEDS = (999101, 999102, 999103)
JITTER = 1e-8
PARAMETER_NAMES = ('sigma_f', 'ell_d', 'ell_p', 'period', 'sigma_n')
BOUNDS = np.array([[0.05, 5.0], [5.0, 500.0], [0.1, 5.0], [8.0, 14.0], [0.01, 2.0]])
LOG_BOUNDS = np.log(BOUNDS)
FAILURE_PENALTY = 1e12
OPTIMIZER_CONFIG = {
    'method': 'L-BFGS-B', 'maxiter': 500, 'maxfun': 5000,
    'ftol': 1e-9, 'gtol': 1e-6, 'cv_jac': '3-point',
    'finite_diff_rel_step': 1e-5, 'continuation_multiplier': 2,
}
# Literal, frozen 8x5 matrix: 5%-95% log-space Latin hypercube, seed 20260920.
STARTS_THETA = np.array([
    [0.48879528, 134.29635454, 0.57492580, 11.00940152, 0.88733950],
    [2.64964466,   7.76278442, 3.68874093, 12.91183230, 0.19192450],
    [0.15531736, 350.23716396, 0.25072509, 11.69159841, 0.03157694],
    [0.18474810,  79.42676569, 0.12626689,  8.37650750, 0.08411781],
    [0.79902856,  10.85979860, 1.08940965,  9.65241411, 0.02319772],
    [0.06306897, 212.30869876, 0.43638883, 10.36895138, 0.73715488],
    [1.35064883,  48.77297737, 2.55673145,  9.06419673, 0.07102522],
    [1.45346138,  24.41659601, 1.20994163, 12.42047826, 0.38035871],
])
STARTS_LOG_THETA = np.log(STARTS_THETA)
assert np.all((STARTS_THETA > BOUNDS[:, 0]) & (STARTS_THETA < BOUNDS[:, 1]))

VERSIONS = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scipy': scipy.__version__, 'sklearn': sklearn.__version__,
    'statsmodels': sm.__version__, 'matplotlib': matplotlib.__version__,
}
print(VERSIONS)
print('Fixed numerical jitter:', JITTER)
print('Parameter bounds:', dict(zip(PARAMETER_NAMES, BOUNDS.tolist())))
print('Optimiser configuration:', OPTIMIZER_CONFIG)
print('Bootstrap:', {'seed': BOOTSTRAP_SEED, 'resamples': BOOTSTRAP_RESAMPLES})
display(pd.DataFrame(STARTS_THETA, columns=PARAMETER_NAMES).rename_axis('start'))

# Persistent storage is established before any scientific computation.
import os, hashlib, time, zipfile
from datetime import datetime, timezone
from IPython.display import display, Markdown

RUN_ID = 'frozen_rerun_01'  # Keep this unchanged when resuming this run.
PROTOCOL_SHA256 = '4efe35f759ecc374477acb894019b1e72628819014c7b4f7266bb0be8bf143b1'
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')
    if not drive_root.is_dir():
        raise RuntimeError('Google Drive must be mounted before experiments run.')
    RUN_ROOT = drive_root / 'A2_GP_LML_vs_CV' / RUN_ID
else:
    RUN_ROOT = Path.cwd() / 'persistent_runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_ROOT)

def atomic_json(destination, payload):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    encoded = json.dumps(payload, indent=2, sort_keys=True, allow_nan=False)
    with temporary.open('w', encoding='utf-8') as handle:
        handle.write(encoded)
        handle.flush()
        os.fsync(handle.fileno())
    # Round-trip verification catches incomplete writes before promotion.
    assert json.loads(temporary.read_text(encoding='utf-8')) == json.loads(encoded)
    temporary.replace(destination)

environment = {'versions': VERSIONS, 'platform': platform.platform(),
               'machine': platform.machine(), 'protocol_sha256': PROTOCOL_SHA256,
               'settings': {'jitter': JITTER, 'bounds': BOUNDS.tolist(),
                            'starts': STARTS_THETA.tolist(), 'optimizer': OPTIMIZER_CONFIG,
                            'seeds': list(SCIENTIFIC_SEEDS),
                            'bootstrap_seed': BOOTSTRAP_SEED,
                            'bootstrap_resamples': BOOTSTRAP_RESAMPLES}}
manifest_path = RUN_ROOT / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if any(manifest['environment'][key] != environment[key]
           for key in ('versions', 'protocol_sha256', 'settings')):
        raise RuntimeError('Resume environment/protocol differs. Preserve this run; restore its '
                           'package versions or choose a NEW RUN_ID for a separate full run. '
                           'Never silently mix environments in one checkpoint set.')
else:
    if any((RUN_ROOT / 'checkpoints').glob('**/*.json')):
        raise RuntimeError('Unmanifested checkpoints found: use a new empty RUN_ID.')
    manifest = {'run_id': RUN_ID, 'created_utc': datetime.now(timezone.utc).isoformat(),
                'environment': environment}
    atomic_json(manifest_path, manifest)
RUN_FINGERPRINT = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
probe = RUN_ROOT / '.write_probe.json'
atomic_json(probe, {'write_test': True})
assert json.loads(probe.read_text())['write_test']
probe.unlink()
print('PERSISTENT RUN DIRECTORY:', RUN_ROOT, flush=True)
print('Drive checkpoint writes verified. Resume with the SAME RUN_ID and Run all.', flush=True)
print('Only the interrupted, unfinished seed may need to be recomputed.', flush=True)

def persistent_batch(items, directory, worker, key, prefix, experiment):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    items = list(items)
    records = []
    started = time.perf_counter()
    durations = []
    for position, item in enumerate(items, 1):
        destination = directory / f'{prefix}_{int(item)}.json'
        if destination.exists():
            record = json.loads(destination.read_text(encoding='utf-8'))
            if not (record.get('complete') is True and record.get(key) == int(item)
                    and record.get('experiment') == experiment
                    and record.get('run_fingerprint') == RUN_FINGERPRINT):
                raise RuntimeError(f'Checkpoint validation failed: {destination}. Preserve it for inspection.')
            records.append(record)
            print(f'[{directory.name}] {position}/{len(items)}: {prefix} {item} loaded from Drive/storage', flush=True)
            continue
        print(f'[{directory.name}] {position}/{len(items)}: computing {prefix} {item} ...', flush=True)
        tick = time.perf_counter()
        record = worker(item)
        assert record.get('complete') is True and record.get(key) == int(item)
        assert record.get('experiment') == experiment
        record['run_fingerprint'] = RUN_FINGERPRINT
        atomic_json(destination, record)
        records.append(record)
        durations.append(time.perf_counter() - tick)
        eta = sum(durations) / len(durations) * (len(items) - position) / 60
        print(f'[{directory.name}] SAVED {position}/{len(items)} | '
              f'elapsed {(time.perf_counter()-started)/60:.1f} min | '
              f'rough remaining {eta:.1f} min', flush=True)
    return records

def current_synthetic_report(summary, label):
    s = summary
    lo, hi = s['bootstrap_95_low'], s['bootstrap_95_high']
    direction = ('CV advantage' if hi < 0 else 'LML advantage' if lo > 0
                 else 'interval includes zero; directional evidence is inconclusive')
    report = (f'### {label}: current-run results\n\n'
              f"Successful pairs: **{s['successful_pairs']}/50**. "
              f"Mean Δ: **{s['mean_delta']:+.6f}**; median: **{s['median_delta']:+.6f}**. "
              f'95% paired-bootstrap interval: **[{lo:+.6f}, {hi:+.6f}]**. '
              f"CV wins: **{s['cv_wins']}**, exact ties: **{s['exact_ties']}**. "
              f'Interpretation: **{direction}** under this frozen generator. '
              'This is a conditional average result, not universal superiority.')
    (RUN_ROOT / f'{label.lower()}_results.md').write_text(report, encoding='utf-8')
    display(Markdown(report))


## 2. Literature motivation, model theory, and numerical policy

### Task and information interfaces

Training receives finite, aligned vectors $\mathbf t\in\mathbb R^n$ and $\mathbf y\in\mathbb R^n$, ordered chronologically, with no missing observations. Real inputs are integer years and annual sunspot numbers from the bundled statsmodels `SUNACTIVITY` series, 1700–2008. Synthetic training times are $0,\ldots,249$. Time stays in years; only real targets are standardized using the training window's mean $m$ and positive standard deviation $s$ (population convention, `ddof=0`). Synthetic fits use $m=0,s=1$.

Either learning criterion selects the five positive hyperparameters $\hat\theta=(\hat\sigma_f,\hat\ell_d,\hat\ell_p,\hat p,\hat\sigma_n)$ within the frozen bounds. The trained state retains training times, the scaler, selected hyperparameters, and the derived Cholesky factor and solve vector. Hyperparameters are selected; the factor and solve vector are computed from them. A constant real training target is rejected because its standard deviation is zero.

Deployment receives eleven strictly future, consecutive time points and no future targets. It returns $\mu_*\in\mathbb R^{11}$ and $\Sigma_*\in\mathbb R^{11\times11}$ specifying an observed-target Gaussian in original target units. Marginal intervals are derived from its diagonal. This is a probabilistic forecasting study, not a claim to predict solar physics or operational space-weather risk.

### Historical development and the question left open

Williams and Rasmussen's original GP regression paper connects function-space Gaussian priors with earlier regression and Kriging methods. It obtains prediction by conditioning a joint Gaussian and studies both optimized and integrated hyperparameters. This establishes an important distinction for this project: exact conditional Gaussian algebra does not make plug-in hyperparameter selection fully Bayesian. Their work also acknowledges multiple optimization optima. [Williams & Rasmussen, NIPS 1995, §§2–3](https://proceedings.neurips.cc/paper_files/paper/1995/file/7cce53cf90577442771720a370c3c723-Paper.pdf).

Rasmussen and Williams subsequently distinguish evidence-based selection from cross-validation and explicitly allow predictive log loss in CV. Their CO2 example uses a periodic covariance multiplied by a squared-exponential decay, with a one-year period, as one part of a larger model. Here the product is the entire latent kernel and its constant period is learned. The evidence maximization used here is type-II maximum likelihood: latent function values are integrated out, but hyperparameters are optimized. [GPML, chapter 5, §§5.2–5.4; equation (5.16), p. 120](https://gaussianprocess.org/gpml/chapters/RW5.pdf).

Bachoc supplies a more specific comparison than a general claim that CV forecasts better. The single-variance analysis assesses how estimated predictive variance reflects conditional prediction error under a fixed correlation model; misspecified cases and extensions to correlation parameters are examined numerically. The correct-specification efficiency result and misspecification experiments motivate our hypotheses. They do not prove them for noisy, regularly spaced, eleven-year joint forecasts with forward CV. In particular, the paper discusses designs where CV performs poorly. [Bachoc, 2013, §§2–5](https://arxiv.org/pdf/1301.4320).

Forecast evaluation needs an additional temporal argument. Tashman discusses rolling origins, recalibration and multiple test periods. Our expanding-history design imitates the information available at deployment; random folds could train on years later than a validation target. This is a task-specific reason for forward validation, not a claim that random CV is invalid for every dependent-data problem. [Tashman, 2000](https://doi.org/10.1016/S0169-2070(00)00065-0).

The research contribution is a reproducible, controlled comparison within one fixed GP family: does matching the selection criterion to future-block density help when only the generating period mechanism changes, and does that conclusion transfer to a historical series? This is an empirical contribution and a precise question, not an assertion of a new kernel, scoring rule, or first-ever ML-versus-CV comparison.

### Domain evidence and why eleven years

SILSO's cycle table reports minimum-to-minimum lengths, including 9 years and 13 years 7 months; ESA gives approximately 9.0–13.5 years. These justify an eleven-year deployment block and the broad, pre-frozen $p\in[8,14]$ as plausible design choices, not exact physical laws. The GP's recurrence parameter is not identical to an observed minimum-to-minimum duration. Boundary hits must therefore be reported. The sources share historical observations and are corroboration rather than independent evidence. [SILSO cycle table](https://www.sidc.be/SILSO/cyclesmm); [ESA, 2007](https://www.esa.int/Science_Exploration/Space_Science/SOHO_-_keeping_an_eye_on_the_Sun_for_12_years).

The numerical series is the 309-observation public-domain statsmodels snapshot sourced from NGDC. SILSO's current duration table uses smoothed monthly Version 2.0 sunspot numbers; it is domain context, not a replacement for the frozen annual data. No claim of equal data versions is made. [statsmodels dataset documentation](https://www.statsmodels.org/stable/datasets/generated/sunspots.html).

The lag-11 baseline asks whether copying a previous cycle is already sufficient. Its forecast covariance is diagonal, with a training-only residual second moment. The GP is studied because it learns temporal dependence and a joint distribution; improved point accuracy over this simple baseline is an empirical question, not an assumption.

### Hypothesis family and expected behaviour

The statistical model is

$$
f\sim\mathcal{GP}(0,k_\theta),\qquad z_i=f(t_i)+\epsilon_i,
\qquad \epsilon_i\overset{\mathrm{iid}}{\sim}\mathcal N(0,\sigma_n^2).
$$

Its hypothesis family is a set of distributions over functions indexed by the bounded hyperparameters, not a finite collection of five-parameter regression curves. The posterior mean depends on all training observations. Our kernel is stationary: it depends on time differences. It has one constant recurrence period; the decay term allows distant cycles to become less alike without specifying a changing physical period. Products preserve positive semidefiniteness. [GPML, chapter 4, §§4.1–4.2](https://gaussianprocess.org/gpml/chapters/RW4.pdf).

At one-period separation the normalized covariance is $\exp[-p^2/(2\ell_d^2)]$; at half-period separation the periodic factor is $\exp[-2/\ell_p^2]$. Thus $\ell_d$ controls persistence across cycles and $\ell_p$ controls within-cycle smoothness. Signal and noise scales distinguish correlated latent variation from independent observation noise. As the test time moves far from all training times with finite $\ell_d$, cross-covariances vanish: the fitted-space mean approaches zero and observed marginal variance approaches $\sigma_f^2+\sigma_n^2$. In real units the mean approaches the training mean. These are analytical expectations, not additional experiments.

### Construction loss, deployment loss and what calibration means

LML minimizes the negative training log density. Blocked CV minimizes the average held-out eleven-dimensional JNLPD over four forward folds. The outer block is used only for evaluation. The logarithmic score is strictly proper on an appropriate class of distributions, and negative log score reverses its orientation so lower is better. It can penalize a confidently misplaced distribution heavily. [Gneiting & Raftery, 2007, §4.1, equation (19)](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jasa.pdf).

For a true conditional density $p$ and proposed density $q$, assuming the expectations exist, the elementary identity

$$
\mathbb E_p\!\left[-\frac1h\log q(Y)\right]
-\mathbb E_p\!\left[-\frac1h\log p(Y)\right]
=\frac1h\operatorname{KL}(p\Vert q)\geq0
$$

explains the ideal comparison for $h=11$. The unknown real conditional distribution prevents its direct calculation. We evaluate realized external scores instead. Under a restricted Gaussian family, propriety does not guarantee that the truth is available or that plug-in prediction is calibrated. Marginal 95% coverage is not 95% simultaneous coverage of the eleven-year trajectory, and correct marginal coverage does not establish correct cross-year dependence. The joint score and the marginal diagnostics answer different questions. Negative JNLPD is possible because a continuous density may exceed one; absolute scores depend on target units.

### Task-specific customizations and computation

The customizations are the eleven-year joint-density CV objective with training-only scaling inside each real fold, and the shared convergence-aware eight-start recovery rule. Covariance, analytic derivatives, conditioning, LML, scoring and fold logic are explicit NumPy/SciPy implementations; sklearn is a fixed-parameter test reference. L-BFGS-B is a library optimizer, not an original implementation. The assessment FAQ permits this division of work, provided the author understands and explains it.

Both methods share bounds, initial points and termination rules, but they do not have equal computational cost. One LML value/gradient involves dense factorization and five matrix-right-hand-side solves, giving $O(n^3)$ time for five fixed parameters and $O(n^2)$ working storage. One CV value sums four factorization/prediction costs; three-point numerical gradients require further objective evaluations. Full-covariance prediction adds $O(n^2h+nh^2+h^3)$ work, including the block density score. E6 measures a narrower factor/solve pipeline; actual optimizer runtimes and evaluation counts supply the end-to-end evidence.

The program never constructs the full inverse. This is a numerical implementation policy, not a claim of reduced asymptotic complexity. Multiple starts reduce initialization sensitivity; they do not certify the global optimum. The methods therefore compare implemented selection procedures as well as their intended objectives.

### Derivation-to-code: exact GP training and forecasting

This section fixes the notation used by both the mathematics and the implementation. Let

$$
\boldsymbol{\theta}
=(\sigma_f,\ell_d,\ell_p,p,\sigma_n),\qquad
d_{ab}=t_a-t_b,\qquad
u_{ab}=\frac{\pi d_{ab}}{p}.
$$

All five parameters are positive and are optimized in log coordinates.

#### 1. Locally periodic covariance and noisy training matrix

The latent-function covariance is

$$
k_f(t_a,t_b;\boldsymbol{\theta})
=\sigma_f^2
\exp\!\left(
-\frac{d_{ab}^2}{2\ell_d^2}
-\frac{2\sin^2(u_{ab})}{\ell_p^2}
\right).
$$

For training times $X=(t_1,\ldots,t_n)$, define $[K_f]_{ab}=k_f(t_a,t_b)$. The observed-target training covariance is

$$
K_y=K_f+\sigma_n^2 I+jI=LL^\top,
$$

where $\sigma_n^2 I$ is observation-noise variance and the fixed $j=10^{-8}$ is numerical jitter. Jitter stabilises the factorisation; it is not treated as forecast uncertainty. `locally_periodic_kernel`, `training_covariance` and `factorize_gp` implement these equations.

#### 2. Gaussian conditioning

Let $\mathbf z$ denote targets in fitted space: standardized targets for real data and the generated targets for synthetic data. The solve vector is

$$
\boldsymbol{\alpha}=K_y^{-1}\mathbf z,
$$

computed by Cholesky solves rather than an explicit inverse. With cross-covariance $K_{*X}^f$ and test covariance $K_{**}^f$, conditioning the joint Gaussian gives

$$
\boldsymbol{\mu}_*^{(z)}=K_{*X}^f\boldsymbol{\alpha},
$$

$$
\Sigma_{*,\mathrm{latent}}^{(z)}
=K_{**}^f-K_{*X}^fK_y^{-1}K_{X*}^f,
$$

and the covariance of future **observations** is

$$
\Sigma_{*,\mathrm{obs}}^{(z)}
=\Sigma_{*,\mathrm{latent}}^{(z)}+\sigma_n^2I.
$$

For a real-data training mean $m$ and scale $s$,

$$
\boldsymbol{\mu}_*^{(y)}=m+s\boldsymbol{\mu}_*^{(z)},
\qquad
\Sigma_*^{(y)}=s^2\Sigma_*^{(z)}.
$$

`gp_predict` performs the conditioning; `unstandardize_prediction` returns real-data forecasts to original sunspot units.

#### 3. Negative log marginal likelihood

Because $K_y=LL^\top$, the implemented negative log marginal likelihood is

$$
\mathcal L(\boldsymbol{\eta})
=\frac12\mathbf z^\top\boldsymbol{\alpha}
+\sum_{i=1}^{n}\log L_{ii}
+\frac n2\log(2\pi),
\qquad
\boldsymbol{\eta}=\log\boldsymbol{\theta}.
$$

The three terms are, respectively, data fit, covariance-volume complexity and the Gaussian normalising constant. LML training minimizes $\mathcal L$.

For

$$
D_j=\frac{\partial K_y}{\partial\eta_j},
$$

the analytic negative-LML gradient is

$$
\frac{\partial\mathcal L}{\partial\eta_j}
=\frac12\left[
\operatorname{tr}(K_y^{-1}D_j)
-\boldsymbol{\alpha}^\top D_j\boldsymbol{\alpha}
\right].
$$

The implemented covariance derivatives are

$$
\begin{aligned}
\frac{\partial K_y}{\partial\log\sigma_f}
&=2K_f,\\[2mm]
\frac{\partial K_y}{\partial\log\ell_d}
&=K_f\odot\frac{D^2}{\ell_d^2},\\[2mm]
\frac{\partial K_y}{\partial\log\ell_p}
&=K_f\odot\frac{4\sin^2(U)}{\ell_p^2},\\[2mm]
\frac{\partial K_y}{\partial\log p}
&=K_f\odot\frac{2U\sin(2U)}{\ell_p^2},\\[2mm]
\frac{\partial K_y}{\partial\log\sigma_n}
&=2\sigma_n^2I,
\end{aligned}
$$

where $D=[d_{ab}]$, $U=[u_{ab}]$ and $\odot$ denotes elementwise multiplication. `kernel_log_derivatives` returns these matrices in the displayed order. `negative_lml_and_grad` evaluates the trace through Cholesky solves and the quadratic term through matrix–vector products. No full $K_y^{-1}$ is constructed, although these matrix right-hand-side solves still have dense-GP time and storage costs.

#### 4. Full-block joint predictive score

Let $h=11$, residual $\mathbf r=\mathbf y_*-\boldsymbol{\mu}_*$ and

$$
\Sigma_{*,\mathrm{obs}}=CC^\top.
$$

The primary deployment loss is the joint negative log predictive density per forecast observation:

$$
\operatorname{JNLPD}
=-\frac1h\log\mathcal N
\!\left(\mathbf y_*;\boldsymbol{\mu}_*,\Sigma_{*,\mathrm{obs}}\right)
=\frac{
\lVert C^{-1}\mathbf r\rVert_2^2
+2\sum_{i=1}^{h}\log C_{ii}
+h\log(2\pi)
}{2h}.
$$

`joint_nlpd` computes this with a Cholesky factor of the full predictive covariance. The score therefore evaluates the joint distribution—including cross-year dependence—rather than summing independently scored marginal densities. Natural logarithms give nats per forecast observation.

#### 5. Equation-to-code map

| Mathematical quantity | Implementation | Verification |
|---|---|---|
| $k_f$, $K_f$, $K_y$ | `locally_periodic_kernel`, `training_covariance` | symmetry, PSD and sklearn kernel checks |
| $L$, $\boldsymbol{\alpha}$ | `factorize_gp` | Cholesky-based E1 comparisons |
| $\boldsymbol{\mu}_*$, $\Sigma_*$ | `gp_predict` | sklearn mean/covariance and noise-once checks |
| $\mathcal L$, $\nabla_{\boldsymbol{\eta}}\mathcal L$ | `negative_lml_and_grad` | central differences and mapped sklearn gradients |
| JNLPD | `joint_nlpd` | SciPy multivariate-normal reference |
| Deployment-shaped selection | `blocked_cv_objective` | leakage, fold-scaling and original-unit checks |

E1 is the correctness gate for this mapping. Scikit-learn stores amplitude and noise as variances, so its corresponding log-gradient entries are multiplied by two when compared with this notebook's log-standard-deviation coordinates.

In [ ]:
def as_time_vector(t):
    return np.asarray(t, dtype=float).reshape(-1)

def unpack_log_theta(log_theta):
    log_theta = np.asarray(log_theta, dtype=float)
    if log_theta.shape != (5,):
        raise ValueError('log_theta must contain five log-parameters')
    return np.exp(log_theta)

def locally_periodic_kernel(t1, t2, theta):
    sigma_f, ell_d, ell_p, period, _ = np.asarray(theta, dtype=float)
    t1, t2 = as_time_vector(t1), as_time_vector(t2)
    d = t1[:, None] - t2[None, :]
    u = np.pi * d / period
    exponent = -(d ** 2) / (2.0 * ell_d ** 2) - 2.0 * np.sin(u) ** 2 / ell_p ** 2
    return sigma_f ** 2 * np.exp(exponent)

def kernel_log_derivatives(t, theta):
    sigma_f, ell_d, ell_p, period, sigma_n = np.asarray(theta, dtype=float)
    t = as_time_vector(t)
    d = t[:, None] - t[None, :]
    u = np.pi * d / period
    Kf = locally_periodic_kernel(t, t, theta)
    I = np.eye(t.size)
    return np.stack([
        2.0 * Kf,
        Kf * d ** 2 / ell_d ** 2,
        Kf * 4.0 * np.sin(u) ** 2 / ell_p ** 2,
        Kf * 2.0 * u * np.sin(2.0 * u) / ell_p ** 2,
        2.0 * sigma_n ** 2 * I,
    ], axis=0)

def training_covariance(t, theta, jitter=JITTER):
    sigma_n = float(theta[4])
    return locally_periodic_kernel(t, t, theta) + (sigma_n ** 2 + jitter) * np.eye(len(as_time_vector(t)))

def factorize_gp(t, y, theta, jitter=JITTER):
    t, y = as_time_vector(t), np.asarray(y, dtype=float).reshape(-1)
    if t.size != y.size:
        raise ValueError('t and y must have the same length')
    Ky = training_covariance(t, theta, jitter)
    factor = cho_factor(Ky, lower=True, check_finite=True)
    alpha = cho_solve(factor, y, check_finite=True)
    return Ky, factor, alpha

def gp_predict(t_train, y_train, t_test, theta, observed=True, jitter=JITTER):
    t_train, t_test = as_time_vector(t_train), as_time_vector(t_test)
    _, factor, alpha = factorize_gp(t_train, y_train, theta, jitter)
    Kxs = locally_periodic_kernel(t_test, t_train, theta)
    mean = Kxs @ alpha
    solved = cho_solve(factor, Kxs.T, check_finite=True)
    covariance = locally_periodic_kernel(t_test, t_test, theta) - Kxs @ solved
    covariance = 0.5 * (covariance + covariance.T)
    if observed:
        covariance = covariance + float(theta[4]) ** 2 * np.eye(t_test.size)
    return mean, covariance

def negative_lml_and_grad(log_theta, t, y, jitter=JITTER):
    theta = unpack_log_theta(log_theta)
    t, y = as_time_vector(t), np.asarray(y, dtype=float).reshape(-1)
    Ky, factor, alpha = factorize_gp(t, y, theta, jitter)
    L = np.tril(factor[0])
    log_det = 2.0 * np.log(np.diag(L)).sum()
    nll = 0.5 * y @ alpha + 0.5 * log_det + 0.5 * y.size * np.log(2.0 * np.pi)
    derivatives = kernel_log_derivatives(t, theta)
    quadratic_terms = np.array([alpha @ derivative @ alpha for derivative in derivatives])
    trace_terms = np.array([np.trace(cho_solve(factor, derivative, check_finite=True))
                            for derivative in derivatives])
    grad = 0.5 * (trace_terms - quadratic_terms)  # gradient of negative LML
    return float(nll), grad

def joint_nlpd(y, mean, covariance, per_observation=True):
    y = np.asarray(y, dtype=float).reshape(-1)
    mean = np.asarray(mean, dtype=float).reshape(-1)
    covariance = np.asarray(covariance, dtype=float)
    if y.shape != mean.shape or covariance.shape != (y.size, y.size):
        raise ValueError('Incompatible joint-density shapes')
    L = cholesky(covariance, lower=True, check_finite=True)
    residual_whitened = solve_triangular(L, y - mean, lower=True, check_finite=True)
    value = 0.5 * (residual_whitened @ residual_whitened)
    value += np.log(np.diag(L)).sum() + 0.5 * y.size * np.log(2.0 * np.pi)
    return float(value / y.size if per_observation else value)

## 3. Standardisation, scoring units, and split construction

For each real training window, $z=(y-m)/s$ with training-only $m$ and $s$. An $h$-dimensional change of variables gives

$$\log p_y(\mathbf y)=\log p_z(\mathbf z)-h\log s,
\qquad\mathrm{JNLPD}_y=\mathrm{JNLPD}_z+\log s.$$

This constant does not change an inner-fold optimization ranking, but original-unit scores are required for comparing origins and the baseline. The code converts predictions back before scoring. The same numerical standardized hyperparameters are evaluated across folds; their signal/noise scales in original units depend on each fold's scaler. After CV selection, the posterior is recomputed using the full outer-training scaler. Future observations never determine a scaler. Synthetic folds retain generated targets without standardization.

For E2/E3 the four validation blocks are 206–216, 217–227, 228–238 and 239–249; each training block starts at 0 and ends immediately before its validation block. The external block is 250–260. For the first real origin (1899), inner validation blocks are 1855–1865, 1866–1876, 1877–1887 and 1888–1898; the external block is 1899–1909. Later real origins repeat this construction with expanding history. There is no nested optimizer inside each CV objective evaluation: candidate hyperparameters are shared across the four fold posteriors.

Synthetic uncertainty intervals resample complete seed-level paired differences, preserving the LML/CV pairing. They describe Monte Carlo uncertainty for the frozen generator, conditional on successful pairs, not uncertainty over arbitrary real-world series. Report exclusions and the frozen failure sensitivity. Non-overlapping real test blocks still share histories and temporal dependence; they are descriptive rather than independent replications.

In [ ]:
def fit_standardizer(y_train):
    y_train = np.asarray(y_train, dtype=float)
    mean, scale = float(y_train.mean()), float(y_train.std(ddof=0))
    if not np.isfinite(scale) or scale <= 0.0:
        raise ValueError('Training scale must be positive and finite')
    return mean, scale

def standardize(y, mean, scale):
    return (np.asarray(y, dtype=float) - mean) / scale

def unstandardize_prediction(mean_z, covariance_z, mean_y, scale_y):
    return mean_y + scale_y * np.asarray(mean_z), scale_y ** 2 * np.asarray(covariance_z)

def inner_cv_blocks(outer_origin_index):
    T = int(outer_origin_index)
    validation = [np.arange(T - 44, T - 33), np.arange(T - 33, T - 22),
                  np.arange(T - 22, T - 11), np.arange(T - 11, T)]
    folds = []
    for valid in validation:
        train = np.arange(0, int(valid[0]))
        if train.size == 0 or train[-1] >= valid[0]:
            raise ValueError('Invalid expanding-window fold')
        folds.append((train, valid))
    return folds

SUNSPOT_FIRST_YEAR = 1700

def year_to_index(year, first_year=SUNSPOT_FIRST_YEAR):
    year = int(year)
    index = year - int(first_year)
    if index < 0:
        raise ValueError('year precedes the series')
    return index

def inner_cv_blocks_from_year(outer_start_year):
    return inner_cv_blocks(year_to_index(outer_start_year))

def blocked_cv_objective(log_theta, t, y, folds, standardize_target):
    # False for E2/E3; True for every real-data/sunspot fit.
    theta = unpack_log_theta(log_theta)
    t, y = as_time_vector(t), np.asarray(y, dtype=float).reshape(-1)
    scores = []
    try:
        for train_idx, valid_idx in folds:
            t_train, t_valid = t[train_idx], t[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]
            if standardize_target:
                train_mean, train_scale = fit_standardizer(y_train)
                z_train = standardize(y_train, train_mean, train_scale)
                mean_z, covariance_z = gp_predict(t_train, z_train, t_valid, theta, observed=True)
                mean_pred, covariance_pred = unstandardize_prediction(
                    mean_z, covariance_z, train_mean, train_scale)
            else:
                mean_pred, covariance_pred = gp_predict(
                    t_train, y_train, t_valid, theta, observed=True)
            scores.append(joint_nlpd(y_valid, mean_pred, covariance_pred))
    except (np.linalg.LinAlgError, ValueError, FloatingPointError):
        return FAILURE_PENALTY
    value = float(np.mean(scores))
    return value if np.isfinite(value) else FAILURE_PENALTY

def _minimize_once(objective, x0, analytic_gradient, budget_multiplier=1):
    options = {
        'maxiter': OPTIMIZER_CONFIG['maxiter'] * budget_multiplier,
        'maxfun': OPTIMIZER_CONFIG['maxfun'] * budget_multiplier,
        'ftol': OPTIMIZER_CONFIG['ftol'], 'gtol': OPTIMIZER_CONFIG['gtol'],
    }
    jac = True if analytic_gradient else OPTIMIZER_CONFIG['cv_jac']
    if not analytic_gradient:
        options['finite_diff_rel_step'] = OPTIMIZER_CONFIG['finite_diff_rel_step']
    return minimize(objective, np.asarray(x0), method=OPTIMIZER_CONFIG['method'],
                    jac=jac, bounds=LOG_BOUNDS, options=options)

def _optimisation_record(result, start_id, stage):
    within_bounds = bool(np.all(result.x >= LOG_BOUNDS[:, 0]) and
                         np.all(result.x <= LOG_BOUNDS[:, 1]))
    admissible = bool(np.isfinite(result.fun) and result.fun < FAILURE_PENALTY and within_bounds)
    theta = np.exp(result.x)
    span = BOUNDS[:, 1] - BOUNDS[:, 0]
    boundary = (theta - BOUNDS[:, 0] <= 1e-4 * span) | (BOUNDS[:, 1] - theta <= 1e-4 * span)
    return {
        'start_id': int(start_id), 'stage': stage, 'objective': float(result.fun),
        'success': bool(result.success), 'admissible': admissible,
        'message': str(result.message), 'nit': int(result.nit), 'nfev': int(result.nfev),
        'log_theta': result.x.copy(), 'theta': theta, 'boundary_flags': boundary,
    }

def multistart_optimize(objective, analytic_gradient, starts=STARTS_LOG_THETA):
    initial_pairs, records = [], []
    for start_id, x0 in enumerate(np.asarray(starts)):
        try:
            result = _minimize_once(objective, x0, analytic_gradient)
            record = _optimisation_record(result, start_id, 'initial')
            initial_pairs.append((result, record)); records.append(record)
        except Exception as exc:
            records.append({'start_id': start_id, 'stage': 'initial', 'objective': np.inf,
                            'success': False, 'admissible': False, 'message': repr(exc),
                            'nit': 0, 'nfev': 0, 'log_theta': np.full(5, np.nan),
                            'theta': np.full(5, np.nan), 'boundary_flags': np.zeros(5, dtype=bool)})
    converged = [(r, rec) for r, rec in initial_pairs if rec['admissible'] and rec['success']]
    best_converged_value = min((rec['objective'] for _, rec in converged), default=np.inf)
    nonconverged = [(r, rec) for r, rec in initial_pairs
                    if rec['admissible'] and not rec['success']]
    if nonconverged:
        candidate, candidate_record = min(nonconverged, key=lambda item: item[1]['objective'])
        if candidate_record['objective'] < best_converged_value:
            try:
                continued = _minimize_once(objective, candidate.x, analytic_gradient,
                                           OPTIMIZER_CONFIG['continuation_multiplier'])
                continued_record = _optimisation_record(continued, candidate_record['start_id'], 'continuation')
                records.append(continued_record)
                if continued_record['admissible'] and continued_record['success']:
                    converged.append((continued, continued_record))
            except Exception as exc:
                records.append({'start_id': candidate_record['start_id'], 'stage': 'continuation',
                                'objective': np.inf, 'success': False, 'admissible': False,
                                'message': repr(exc), 'nit': 0, 'nfev': 0,
                                'log_theta': np.full(5, np.nan), 'theta': np.full(5, np.nan),
                                'boundary_flags': np.zeros(5, dtype=bool)})
    if not converged:
        return {'success': False, 'result': None, 'theta': None, 'records': records}
    best_result, best_record = min(converged, key=lambda item: item[1]['objective'])
    return {'success': True, 'result': best_result, 'theta': np.exp(best_result.x),
            'records': records, 'best_record': best_record}

def seasonal_naive_predictive(train_years, train_y, test_years, lag=11):
    train_years = np.asarray(train_years, dtype=int)
    train_y = np.asarray(train_y, dtype=float)
    test_years = np.asarray(test_years, dtype=int)
    if train_y.size <= lag:
        raise ValueError('Insufficient history for lag-11 baseline')
    residuals = train_y[lag:] - train_y[:-lag]
    variance = float(np.mean(residuals ** 2))  # frozen zero-mean residual MLE
    lookup = dict(zip(train_years.tolist(), train_y.tolist()))
    try:
        mean = np.array([lookup[int(year - lag)] for year in test_years])
    except KeyError as exc:
        raise ValueError('A required lagged target is unavailable') from exc
    return mean, variance * np.eye(test_years.size), variance

def absolute_log_parameter_error(theta_hat, theta_true):
    theta_hat, theta_true = np.asarray(theta_hat, dtype=float), np.asarray(theta_true, dtype=float)
    if np.any(theta_hat <= 0.0) or np.any(theta_true <= 0.0):
        raise ValueError('Log-parameter recovery requires positive parameters')
    return np.abs(np.log(theta_hat / theta_true))

def paired_bootstrap_mean_interval(deltas, confidence=0.95,
                                   seed=BOOTSTRAP_SEED, resamples=BOOTSTRAP_RESAMPLES):
    deltas = np.asarray(deltas, dtype=float)
    deltas = deltas[np.isfinite(deltas)]
    if deltas.size == 0:
        raise ValueError('No successful paired deltas')
    rng = np.random.default_rng(seed)
    sampled = rng.choice(deltas, size=(resamples, deltas.size), replace=True).mean(axis=1)
    alpha = 1.0 - confidence
    return tuple(np.quantile(sampled, [alpha / 2.0, 1.0 - alpha / 2.0]))

def cv_win_summary(deltas, confidence=0.95):
    deltas = np.asarray(deltas, dtype=float)
    deltas = deltas[np.isfinite(deltas)]
    n = int(deltas.size)
    if n == 0:
        raise ValueError('No successful paired deltas')
    wins, ties = int(np.sum(deltas < 0.0)), int(np.sum(deltas == 0.0))
    proportion = wins / n
    z = float(norm.ppf(0.5 + confidence / 2.0))
    denominator = 1.0 + z ** 2 / n
    centre = (proportion + z ** 2 / (2.0 * n)) / denominator
    half_width = z * np.sqrt(proportion * (1.0 - proportion) / n + z ** 2 / (4.0 * n ** 2)) / denominator
    return {'successful_pairs': n, 'cv_wins': wins, 'exact_ties': ties,
            'cv_win_proportion': proportion, 'wilson_low': centre - half_width,
            'wilson_high': centre + half_width}

# E1 — Correctness gate

All assertions below must pass on the final implementation. Thresholds were frozen before this run: kernel minimum eigenvalue at least approximately -1e-10; scikit-learn mean/covariance/LML absolute differences below 1e-6; analytic-gradient relative error below 1e-4.

In [ ]:
E1_RESULTS = []

def record_check(name, value, threshold, passed):
    E1_RESULTS.append({'check': name, 'value': float(value), 'threshold': threshold, 'passed': bool(passed)})
    assert passed, f'E1 FAILED: {name}; value={value}; threshold={threshold}'

rng = np.random.default_rng(E1_SEED)
t_train = np.linspace(0.0, 60.0, 61)
t_test = np.linspace(60.5, 71.0, 11)
theta_ref = np.array([1.10, 35.0, 0.90, 10.8, 0.20])
Kf_ref = locally_periodic_kernel(t_train, t_train, theta_ref)
y_train = rng.multivariate_normal(np.zeros(t_train.size), training_covariance(t_train, theta_ref, jitter=0.0))

symmetry_error = np.max(np.abs(Kf_ref - Kf_ref.T))
minimum_eigenvalue = np.linalg.eigvalsh(Kf_ref).min()
kernel_diagonal_error = np.max(np.abs(np.diag(Kf_ref) - theta_ref[0] ** 2))
training_diagonal_error = np.max(np.abs(np.diag(training_covariance(t_train, theta_ref))
                                            - np.diag(Kf_ref) - theta_ref[4] ** 2 - JITTER))
record_check('kernel symmetry max error', symmetry_error, '< 1e-12', symmetry_error < 1e-12)
record_check('kernel minimum eigenvalue', minimum_eigenvalue, '>= -1e-10', minimum_eigenvalue >= -1e-10)
record_check('clean-kernel diagonal error', kernel_diagonal_error, '< 1e-12', kernel_diagonal_error < 1e-12)
record_check('training diagonal increment error', training_diagonal_error, '< 1e-12', training_diagonal_error < 1e-12)

### E1a. Analytic LML gradient versus central finite differences

This checks the gradient of the implemented **negative** LML with respect to all five log-parameters, including the period derivative.

In [ ]:
def central_difference_gradient(function, x, step=1e-6):
    x = np.asarray(x, dtype=float)
    gradient = np.empty_like(x)
    for k in range(x.size):
        shift = np.zeros_like(x)
        shift[k] = step
        gradient[k] = (function(x + shift) - function(x - shift)) / (2.0 * step)
    return gradient

gradient_test_thetas = [
    theta_ref,
    np.array([0.75, 18.0, 1.40, 9.4, 0.12]),
    np.array([1.65, 90.0, 0.55, 12.6, 0.35]),
]
gradient_rows = []
for case, theta in enumerate(gradient_test_thetas, start=1):
    log_theta = np.log(theta)
    _, analytic = negative_lml_and_grad(log_theta, t_train, y_train)
    numeric = central_difference_gradient(lambda z: negative_lml_and_grad(z, t_train, y_train)[0], log_theta)
    relative_error = np.linalg.norm(analytic - numeric) / max(1.0, np.linalg.norm(analytic), np.linalg.norm(numeric))
    gradient_rows.append({'case': case, 'relative_error': relative_error,
                          **{f'analytic_{name}': analytic[i] for i, name in enumerate(PARAMETER_NAMES)},
                          **{f'numeric_{name}': numeric[i] for i, name in enumerate(PARAMETER_NAMES)}})
    record_check(f'gradient relative error case {case}', relative_error, '< 1e-4', relative_error < 1e-4)
pd.DataFrame(gradient_rows)

### E1b. Independent fixed-hyperparameter comparison with scikit-learn

Scikit-learn differentiates with respect to its stored log variance parameters. Therefore its amplitude and noise gradient entries are multiplied by two to compare with derivatives with respect to `log(sigma_f)` and `log(sigma_n)`. Parameter names are inspected explicitly before mapping.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, ExpSineSquared, WhiteKernel

sigma_f, ell_d, ell_p, period, sigma_n = theta_ref
sk_kernel = (ConstantKernel(sigma_f ** 2, constant_value_bounds='fixed')
             * RBF(ell_d, length_scale_bounds='fixed')
             * ExpSineSquared(ell_p, period, length_scale_bounds='fixed', periodicity_bounds='fixed')
             + WhiteKernel(sigma_n ** 2, noise_level_bounds='fixed'))
sk_gp = GaussianProcessRegressor(kernel=sk_kernel, alpha=JITTER, optimizer=None, normalize_y=False)
sk_gp.fit(t_train[:, None], y_train)
sk_latent_matrix = sk_gp.kernel_.k1(t_train[:, None])
latent_kernel_error = np.max(np.abs(Kf_ref - sk_latent_matrix))
record_check('sklearn latent-kernel max error', latent_kernel_error, '< 1e-6', latent_kernel_error < 1e-6)
sk_mean, sk_cov = sk_gp.predict(t_test[:, None], return_cov=True)
our_mean, our_cov = gp_predict(t_train, y_train, t_test, theta_ref, observed=True)
our_nll, our_nll_grad = negative_lml_and_grad(np.log(theta_ref), t_train, y_train)
sk_lml = sk_gp.log_marginal_likelihood()

mean_error = np.max(np.abs(our_mean - sk_mean))
covariance_error = np.max(np.abs(our_cov - sk_cov))
lml_error = abs(-our_nll - sk_lml)
record_check('sklearn predictive mean max error', mean_error, '< 1e-6', mean_error < 1e-6)
record_check('sklearn predictive covariance max error', covariance_error, '< 1e-6', covariance_error < 1e-6)
record_check('sklearn LML absolute error', lml_error, '< 1e-6', lml_error < 1e-6)
predictive_symmetry_error = np.max(np.abs(our_cov - our_cov.T))
minimum_predictive_variance = np.diag(our_cov).min()
minimum_predictive_eigenvalue = np.linalg.eigvalsh(0.5 * (our_cov + our_cov.T)).min()
record_check('predictive covariance symmetry error', predictive_symmetry_error, '< 1e-12', predictive_symmetry_error < 1e-12)
record_check('minimum predictive variance', minimum_predictive_variance, '>= -1e-10', minimum_predictive_variance >= -1e-10)
record_check('predictive covariance minimum eigenvalue', minimum_predictive_eigenvalue, '>= -1e-10', minimum_predictive_eigenvalue >= -1e-10)

# A second sklearn object leaves hyperparameters free only so eval_gradient=True is available.
sk_free_kernel = (ConstantKernel(sigma_f ** 2, (1e-8, 1e8))
                  * RBF(ell_d, (1e-8, 1e8))
                  * ExpSineSquared(ell_p, period, (1e-8, 1e8), (1e-8, 1e8))
                  + WhiteKernel(sigma_n ** 2, (1e-8, 1e8)))
sk_free = GaussianProcessRegressor(kernel=sk_free_kernel, alpha=JITTER, optimizer=None, normalize_y=False)
sk_free.fit(t_train[:, None], y_train)
_, sk_gradient = sk_free.log_marginal_likelihood(sk_free.kernel_.theta, eval_gradient=True)
sk_names = [hp.name for hp in sk_free.kernel_.hyperparameters]
expected_names = [
    'k1__k1__k1__constant_value', 'k1__k1__k2__length_scale',
    'k1__k2__length_scale', 'k1__k2__periodicity', 'k2__noise_level'
]
assert sk_names == expected_names, f'Unexpected sklearn parameter order: {sk_names}'
mapped_sk_lml_gradient = np.array([2.0 * sk_gradient[0], sk_gradient[1], sk_gradient[2],
                                   sk_gradient[3], 2.0 * sk_gradient[4]])
gradient_mapping_error = np.max(np.abs(our_nll_grad + mapped_sk_lml_gradient))
record_check('sklearn mapped NLL-gradient max error', gradient_mapping_error, '< 1e-6', gradient_mapping_error < 1e-6)
pd.DataFrame({'sklearn_name': sk_names, 'mapped_LML_gradient': mapped_sk_lml_gradient,
              'our_NLL_gradient': our_nll_grad})

### E1c. Observation noise exactly once and JNLPD reference check

In [ ]:
_, latent_cov = gp_predict(t_train, y_train, t_test, theta_ref, observed=False)
noise_increment = our_cov - latent_cov
noise_error = np.max(np.abs(noise_increment - sigma_n ** 2 * np.eye(t_test.size)))
record_check('observation noise added exactly once', noise_error, '< 1e-12', noise_error < 1e-12)

y_test = rng.multivariate_normal(our_mean, our_cov)
our_score = joint_nlpd(y_test, our_mean, our_cov, per_observation=True)
reference_score = -multivariate_normal.logpdf(y_test, mean=our_mean, cov=our_cov) / y_test.size
jnlpd_error = abs(our_score - reference_score)
record_check('JNLPD versus scipy reference', jnlpd_error, '< 1e-10', jnlpd_error < 1e-10)

### E1d. Split leakage and original-unit density identity

In [ ]:
folds = inner_cv_blocks(200)
expected_validation = [(156, 166), (167, 177), (178, 188), (189, 199)]
actual_validation = [(int(v[0]), int(v[-1])) for _, v in folds]
folds_valid = actual_validation == expected_validation and all(
    train.max() < valid.min() and len(valid) == 11 for train, valid in folds)
record_check('inner-fold boundary/leakage check', int(folds_valid), '== 1', folds_valid)
year_folds = inner_cv_blocks_from_year(1899)
actual_year_ranges = [(SUNSPOT_FIRST_YEAR + int(v[0]), SUNSPOT_FIRST_YEAR + int(v[-1]))
                      for _, v in year_folds]
expected_year_ranges = [(1855, 1865), (1866, 1876), (1877, 1887), (1888, 1898)]
year_semantics_valid = actual_year_ranges == expected_year_ranges
record_check('year-to-index fold semantics', int(year_semantics_valid), '== 1', year_semantics_valid)

toy_train, toy_future = np.array([1.0, 2.0, 3.0]), np.array([1000.0])
toy_mean, toy_scale = fit_standardizer(toy_train)
standardizer_error = max(abs(toy_mean - 2.0), abs(toy_scale - np.sqrt(2.0 / 3.0)))
record_check('training-only standardisation check', standardizer_error, '< 1e-12', standardizer_error < 1e-12)

scale = 7.3
shift = 42.0
mean_y, cov_y, y_y = shift + scale * our_mean, scale ** 2 * our_cov, shift + scale * y_test
score_y_direct = joint_nlpd(y_y, mean_y, cov_y)
score_y_identity = our_score + np.log(scale)
units_error = abs(score_y_direct - score_y_identity)
record_check('original-unit JNLPD identity', units_error, '< 1e-10', units_error < 1e-10)

### E1 gate summary

The following cell raises if any check failed. A green PASS is the only permission to proceed to E2–E6.

In [ ]:
e1_table = pd.DataFrame(E1_RESULTS)
display(e1_table)
assert len(e1_table) == 21, f'Expected 21 E1 checks, found {len(e1_table)}'
assert e1_table['passed'].all()
print(f'✅ E1 CORRECTNESS GATE PASSED: {len(e1_table)}/{len(e1_table)} checks passed.')

# Frozen experiments and results

The cells below execute the frozen protocol and display results for the current persistent run. Validated completed checkpoints are reused after interruption. The historical local and first-Colab outputs remain separate records; new results are not required to reproduce their exact floating-point values. Complete every frozen seed regardless of interim outcomes.

## 4. Data audit — annual sunspot number, 1700–2008

Load the public-domain statsmodels dataset; verify 309 consecutive annual observations, missingness, non-negativity, and descriptive properties. Do not use outer-test outcomes for redesign.

In [ ]:
sunspots = sm.datasets.sunspots.load_pandas().data.copy()
sunspots.columns = ['year', 'sunspots']
SUNSPOT_AUDIT = {
    'rows': len(sunspots), 'first_year': int(sunspots.year.min()),
    'last_year': int(sunspots.year.max()), 'missing': int(sunspots.isna().sum().sum()),
    'minimum': float(sunspots.sunspots.min()), 'maximum': float(sunspots.sunspots.max()),
}
assert SUNSPOT_AUDIT == {'rows': 309, 'first_year': 1700, 'last_year': 2008,
                         'missing': 0, 'minimum': 0.0, 'maximum': 190.2}
assert np.array_equal(sunspots.year.to_numpy(), np.arange(1700, 2009))
SUNSPOT_AUDIT

## 5. Optimisation protocol

Implemented above: both criteria use the displayed literal 8x5 starts and frozen L-BFGS-B settings (`maxiter=500`, `maxfun=5000`, `ftol=1e-9`, `gtol=1e-6`). A lower admissible non-converged candidate receives at most one restart from its terminal point with fresh curvature memory and doubled budgets. A still-nonconverged point is diagnostic evidence, not a selected fitted model. If no candidate converges, the method-dataset fit is an optimisation failure. The same rule is applied to LML and CV.

## 6. Blocked CV objective

`blocked_cv_objective(..., standardize_target)` is implemented above. Use `standardize_target=False` for E2/E3 so the correctly specified zero-mean generator remains at its generated scale. Use `standardize_target=True` for sunspots; every fold then recomputes mean/scale from its own training history and scores in original units. CV uses the frozen `jac='3-point'` and `finite_diff_rel_step=1e-5`. The 1e-4/1e-5/1e-6 check is diagnostic only and cannot change this setting after outcomes are seen.

### Pre-E2 engineering smoke test (not scientific evidence)

This uses seed 999001, outside the frozen E2 seeds 0–49, solely to verify the complete eight-start LML → eight-start CV → 11-year forecast pipeline and measure runtime. Its Delta value must not be included in, or used to redesign, E2.

In [ ]:
import time

SMOKE_SEED = 999001
smoke_rng = np.random.default_rng(SMOKE_SEED)
smoke_t = np.arange(261, dtype=float)
smoke_theta_true = np.array([1.0, 50.0, 1.0, 11.0, 0.15])
smoke_cov = training_covariance(smoke_t, smoke_theta_true, jitter=0.0)
smoke_y = smoke_rng.multivariate_normal(np.zeros(smoke_t.size), smoke_cov)
smoke_train, smoke_test = np.arange(250), np.arange(250, 261)
smoke_folds = inner_cv_blocks(250)

start_time = time.perf_counter()
smoke_lml = multistart_optimize(
    lambda z: negative_lml_and_grad(z, smoke_t[smoke_train], smoke_y[smoke_train]),
    analytic_gradient=True)
smoke_lml_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()
smoke_cv = multistart_optimize(
    lambda z: blocked_cv_objective(z, smoke_t, smoke_y, smoke_folds, standardize_target=False),
    analytic_gradient=False)
smoke_cv_seconds = time.perf_counter() - start_time
assert smoke_lml['success'] and smoke_cv['success'], 'Smoke-test optimisation failure; inspect records'

def smoke_outer_score(theta):
    mean, covariance = gp_predict(smoke_t[smoke_train], smoke_y[smoke_train],
                                  smoke_t[smoke_test], theta, observed=True)
    return joint_nlpd(smoke_y[smoke_test], mean, covariance)

smoke_jnlpd_lml = smoke_outer_score(smoke_lml['theta'])
smoke_jnlpd_cv = smoke_outer_score(smoke_cv['theta'])
SMOKE_RESULT = {
    'seed': SMOKE_SEED, 'standardize_target': False,
    'lml_seconds': smoke_lml_seconds, 'cv_seconds': smoke_cv_seconds,
    'lml_jnlpd': smoke_jnlpd_lml, 'cv_jnlpd': smoke_jnlpd_cv,
    'delta_cv_minus_lml': smoke_jnlpd_cv - smoke_jnlpd_lml,
    'lml_converged_candidates': sum(r['success'] and r['admissible'] for r in smoke_lml['records']),
    'cv_converged_candidates': sum(r['success'] and r['admissible'] for r in smoke_cv['records']),
}
pd.Series(SMOKE_RESULT, name='engineering_smoke_only')

### Frozen CV finite-difference-step diagnostic

Before scientific E2, compare relative steps 1e-4, 1e-5, and 1e-6 at the two predeclared interior parameter vectors `(1,50,1,11,0.15)` and `(0.8,40,1.2,10.5,0.2)`, using only smoke seed 999001. The frozen optimisation step remains 1e-5 regardless of this diagnostic.

In [ ]:
def central_relative_gradient(function, x, relative_step):
    x = np.asarray(x, dtype=float)
    h = relative_step * np.maximum(1.0, np.abs(x))
    gradient = np.empty_like(x)
    for j in range(x.size):
        offset = np.zeros_like(x); offset[j] = h[j]
        gradient[j] = (function(x + offset) - function(x - offset)) / (2.0 * h[j])
    return gradient

CV_STEP_DIAGNOSTIC_THETA = (
    np.array([1.0, 50.0, 1.0, 11.0, 0.15]),
    np.array([0.8, 40.0, 1.2, 10.5, 0.2]),
)
CV_STEP_DIAGNOSTIC_STEPS = (1e-4, 1e-5, 1e-6)
cv_smoke_objective = lambda z: blocked_cv_objective(
    z, smoke_t, smoke_y, smoke_folds, standardize_target=False)
diagnostic_rows = []
for point_id, theta in enumerate(CV_STEP_DIAGNOSTIC_THETA, start=1):
    gradients = {step: central_relative_gradient(cv_smoke_objective, np.log(theta), step)
                 for step in CV_STEP_DIAGNOSTIC_STEPS}
    frozen_gradient = gradients[1e-5]
    for step, gradient in gradients.items():
        relative_to_frozen = np.linalg.norm(gradient - frozen_gradient) / max(1.0, np.linalg.norm(frozen_gradient))
        diagnostic_rows.append({
            'point': point_id, 'relative_step': step,
            'relative_to_frozen_1e-5': relative_to_frozen,
            **{f'grad_{name}': gradient[j] for j, name in enumerate(PARAMETER_NAMES)},
        })
CV_STEP_DIAGNOSTIC = pd.DataFrame(diagnostic_rows)
CV_STEP_DIAGNOSTIC

## 7. Probabilistic lag-11 seasonal-naive baseline

Implemented above. The predictive mean is `y[t-11]`. With training residuals `e_t=y_t-y[t-11]`, the frozen variance estimator is the zero-mean Gaussian MLE `sigma_e^2 = mean(e_t^2)`—not `np.var(e)`, which would subtract a residual mean inconsistent with the fixed baseline mean. The 11x11 predictive covariance is diagonal, so this baseline receives no credit for modelling cross-year forecast correlation.

## 8. E2 — correctly specified synthetic experiment (50 seeds)

Use scientific seeds 0–49 and the frozen locally periodic data-generating parameters/indices. Do not standardise each realization. The paired percentile-bootstrap mean interval uses seed 20260922 and 10,000 resamples. CV win proportion is `count(Delta < 0) / successful_pairs`, with a Wilson interval; exact zero ties are reported separately. Hyperparameter recovery is `abs(log(theta_hat/theta_true))`, summarised by median fitted value, median/IQR log-error, and boundary-hit count. JNLPD remains primary.

### Checkpointed E2 runner and three-seed engineering test

Every seed is written atomically after completion. Existing complete checkpoints are loaded and skipped. Engineering seeds 999101–999103 test batching and recovery only; scientific E2 is exclusively seeds 0–49. Hyperparameter recovery is `abs(log(theta_hat/theta_true))`; report median fitted value, median/IQR log-error, and boundary-hit count, while retaining JNLPD as the primary criterion.

In [ ]:
def _json_ready(value):
    if isinstance(value, dict):
        return {str(k): _json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return _json_ready(value.tolist())
    if isinstance(value, np.generic):
        return _json_ready(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

assert _json_ready({'inf': np.inf, 'nan_array': np.array([np.nan])}) == {
    'inf': None, 'nan_array': [None]}

def _fit_checkpoint_summary(fit):
    records = [_json_ready(record) for record in fit['records']]
    if not fit['success']:
        return {'success': False, 'theta': None, 'objective': None, 'winning_start_id': None,
                'boundary_flags': None, 'continuation_used': any(r['stage'] == 'continuation' for r in records),
                'optimizer_message': None, 'records': records}
    best = fit['best_record']
    return {'success': True, 'theta': _json_ready(fit['theta']),
            'objective': float(best['objective']), 'winning_start_id': int(best['start_id']),
            'boundary_flags': _json_ready(best['boundary_flags']),
            'continuation_used': any(r['stage'] == 'continuation' for r in records),
            'optimizer_message': str(best['message']), 'records': records}

def simulate_well_specified(seed):
    rng = np.random.default_rng(int(seed))
    t = np.arange(261, dtype=float)
    theta_true = np.array([1.0, 50.0, 1.0, 11.0, 0.15])
    covariance = training_covariance(t, theta_true, jitter=0.0)
    return t, rng.multivariate_normal(np.zeros(t.size), covariance), theta_true

def run_well_specified_seed(seed):
    t, y, theta_true = simulate_well_specified(seed)
    train_idx, test_idx, folds = np.arange(250), np.arange(250, 261), inner_cv_blocks(250)
    started = time.perf_counter()
    lml_started = time.perf_counter()
    lml_fit = multistart_optimize(
        lambda z: negative_lml_and_grad(z, t[train_idx], y[train_idx]), analytic_gradient=True)
    lml_seconds = time.perf_counter() - lml_started
    cv_started = time.perf_counter()
    cv_fit = multistart_optimize(
        lambda z: blocked_cv_objective(z, t, y, folds, standardize_target=False),
        analytic_gradient=False)
    cv_seconds = time.perf_counter() - cv_started

    def evaluate_fit(fit):
        if not fit['success']:
            return None
        mean, covariance = gp_predict(t[train_idx], y[train_idx], t[test_idx],
                                      fit['theta'], observed=True)
        std = np.sqrt(np.maximum(np.diag(covariance), 0.0))
        metrics = {'jnlpd': joint_nlpd(y[test_idx], mean, covariance),
                   'rmse': float(np.sqrt(np.mean((y[test_idx] - mean) ** 2))),
                   'predictive_mean': mean.tolist(),
                   'predictive_covariance': covariance.tolist()}
        for level in (0.80, 0.90, 0.95):
            z_value = float(norm.ppf(0.5 + level / 2.0))
            lower, upper = mean - z_value * std, mean + z_value * std
            label = str(int(level * 100))
            metrics[f'coverage_{label}'] = float(np.mean((y[test_idx] >= lower) & (y[test_idx] <= upper)))
            metrics[f'mean_width_{label}'] = float(np.mean(upper - lower))
        return metrics

    lml_metrics, cv_metrics = evaluate_fit(lml_fit), evaluate_fit(cv_fit)
    lml_jnlpd = None if lml_metrics is None else lml_metrics['jnlpd']
    cv_jnlpd = None if cv_metrics is None else cv_metrics['jnlpd']
    result = {
        'complete': True, 'experiment': 'well_specified', 'seed': int(seed),
        'standardize_target': False, 'theta_true': theta_true.tolist(),
        'lml': _fit_checkpoint_summary(lml_fit), 'cv': _fit_checkpoint_summary(cv_fit),
        'test_targets': y[test_idx].tolist(), 'lml_metrics': lml_metrics, 'cv_metrics': cv_metrics,
        'lml_jnlpd': lml_jnlpd, 'cv_jnlpd': cv_jnlpd,
        'delta_cv_minus_lml': None if lml_jnlpd is None or cv_jnlpd is None else cv_jnlpd - lml_jnlpd,
        'lml_log_parameter_error': None if not lml_fit['success'] else absolute_log_parameter_error(lml_fit['theta'], theta_true).tolist(),
        'cv_log_parameter_error': None if not cv_fit['success'] else absolute_log_parameter_error(cv_fit['theta'], theta_true).tolist(),
        'lml_seconds': lml_seconds, 'cv_seconds': cv_seconds,
        'total_seconds': time.perf_counter() - started,
    }
    return _json_ready(result)

def run_checkpointed_well_specified(seeds, checkpoint_directory):
    return persistent_batch(seeds, checkpoint_directory, run_well_specified_seed,
                            'seed', 'seed', 'well_specified')

engineering_results = run_checkpointed_well_specified(
    ENGINEERING_SEEDS, Path('checkpoints') / 'engineering_e2')
pd.DataFrame([{
    'seed': r['seed'], 'lml_success': r['lml']['success'], 'cv_success': r['cv']['success'],
    'delta': r['delta_cv_minus_lml'], 'lml_seconds': r['lml_seconds'],
    'cv_seconds': r['cv_seconds'], 'checkpoint_complete': r['complete']
} for r in engineering_results])

### Frozen scientific E2 run — seeds 0–49

This cell runs the complete frozen set without early stopping. Each result is checkpointed immediately. Primary summaries use only successful paired fits; method-specific failures and conservative failure-as-loss counts are reported separately.

In [ ]:
def summarize_e2(results):
    successful = [r for r in results if r['lml']['success'] and r['cv']['success']
                  and r['delta_cv_minus_lml'] is not None]
    deltas = np.array([r['delta_cv_minus_lml'] for r in successful], dtype=float)
    delta_frame = pd.DataFrame({
        'seed': [r['seed'] for r in successful],
        'lml_jnlpd': [r['lml_jnlpd'] for r in successful],
        'cv_jnlpd': [r['cv_jnlpd'] for r in successful],
        'delta_cv_minus_lml': deltas,
    })
    win = cv_win_summary(deltas)
    bootstrap_low, bootstrap_high = paired_bootstrap_mean_interval(deltas)
    summary = {
        'requested_seeds': len(results), 'successful_pairs': len(successful),
        'mean_delta': float(np.mean(deltas)), 'median_delta': float(np.median(deltas)),
        'bootstrap_95_low': float(bootstrap_low), 'bootstrap_95_high': float(bootstrap_high),
        **win,
        'lml_failures': int(sum(not r['lml']['success'] for r in results)),
        'cv_failures': int(sum(not r['cv']['success'] for r in results)),
        'both_failures_unresolved': int(sum(not r['lml']['success'] and not r['cv']['success'] for r in results)),
        'failed_cv_counted_as_cv_losses': int(sum(r['lml']['success'] and not r['cv']['success'] for r in results)),
        'failed_lml_counted_as_lml_losses': int(sum(not r['lml']['success'] and r['cv']['success'] for r in results)),
        'lml_continuations': int(sum(r['lml']['continuation_used'] for r in results)),
        'cv_continuations': int(sum(r['cv']['continuation_used'] for r in results)),
        'total_lml_seconds': float(sum(r['lml_seconds'] for r in results)),
        'total_cv_seconds': float(sum(r['cv_seconds'] for r in results)),
    }
    for method in ('lml', 'cv'):
        method_results = [r for r in results if r[method]['success']]
        metrics = [r[f'{method}_metrics'] for r in method_results]
        summary[f'{method}_mean_rmse'] = float(np.mean([m['rmse'] for m in metrics]))
        summary[f'{method}_median_rmse'] = float(np.median([m['rmse'] for m in metrics]))
        for level in ('80', '90', '95'):
            summary[f'{method}_coverage_{level}'] = float(np.mean([m[f'coverage_{level}'] for m in metrics]))
            summary[f'{method}_mean_width_{level}'] = float(np.mean([m[f'mean_width_{level}'] for m in metrics]))

    recovery_rows = []
    for method in ('lml', 'cv'):
        fitted = np.array([r[method]['theta'] for r in results if r[method]['success']], dtype=float)
        errors = np.array([r[f'{method}_log_parameter_error'] for r in results if r[method]['success']], dtype=float)
        boundaries = np.array([r[method]['boundary_flags'] for r in results if r[method]['success']], dtype=bool)
        for j, name in enumerate(PARAMETER_NAMES):
            recovery_rows.append({
                'method': method.upper(), 'parameter': name,
                'median_fitted': float(np.median(fitted[:, j])),
                'median_log_error': float(np.median(errors[:, j])),
                'q25_log_error': float(np.quantile(errors[:, j], 0.25)),
                'q75_log_error': float(np.quantile(errors[:, j], 0.75)),
                'boundary_hits': int(boundaries[:, j].sum()),
            })
    return delta_frame, summary, pd.DataFrame(recovery_rows)

e2_results = run_checkpointed_well_specified(
    SCIENTIFIC_SEEDS, Path('checkpoints') / 'e2_well_specified')
assert len(e2_results) == 50 and {r['seed'] for r in e2_results} == set(SCIENTIFIC_SEEDS)
assert all(r['complete'] for r in e2_results)
E2_DELTAS, E2_SUMMARY, E2_RECOVERY = summarize_e2(e2_results)
summary_destination = Path('checkpoints') / 'e2_well_specified' / 'summary.json'
summary_temporary = summary_destination.with_suffix('.json.tmp')
with summary_temporary.open('w', encoding='utf-8') as handle:
    json.dump(_json_ready({'summary': E2_SUMMARY, 'delta_records': E2_DELTAS.to_dict('records'),
                           'recovery': E2_RECOVERY.to_dict('records')}),
              handle, indent=2, sort_keys=True, allow_nan=False)
summary_temporary.replace(summary_destination)
display(pd.Series(E2_SUMMARY, name='E2 frozen summary'))
display(E2_DELTAS)
display(E2_RECOVERY)
current_synthetic_report(E2_SUMMARY, 'E2')


### E2 visual evidence and interpretation guide

The current-run summary above is authoritative. A bootstrap interval wholly above zero supports an average LML advantage under correct specification; wholly below zero contradicts H1; an interval spanning zero is inconclusive. Compare the mean with the median and complete seed distribution before attributing the outcome to a few large differences. Read win counts, coverage, interval width, RMSE and parameter-recovery tables together. No direction of the result is assumed in advance.

In [ ]:
figure_dir = Path('figures'); figure_dir.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

deltas = E2_DELTAS['delta_cv_minus_lml'].to_numpy()
axes[0, 0].hist(deltas, bins=12, color='#4C78A8', edgecolor='white')
axes[0, 0].axvline(0, color='black', linestyle='--', linewidth=1)
axes[0, 0].axvline(E2_SUMMARY['mean_delta'], color='#E45756', linewidth=2, label='Mean')
axes[0, 0].set(title='E2 distribution of Delta', xlabel='JNLPD(CV) - JNLPD(LML)', ylabel='Seeds')
axes[0, 0].legend()

ordered = np.sort(deltas)
axes[0, 1].plot(np.arange(1, len(ordered) + 1), ordered, marker='o', markersize=3)
axes[0, 1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0, 1].text(0.03, 0.96,
                f"Mean Delta = {E2_SUMMARY['mean_delta']:.3f}\n"
                f"95% bootstrap CI for mean = [{E2_SUMMARY['bootstrap_95_low']:.3f}, "
                f"{E2_SUMMARY['bootstrap_95_high']:.3f}]",
                transform=axes[0, 1].transAxes, va='top',
                bbox={'boxstyle': 'round', 'facecolor': '#F2CF5B', 'alpha': 0.3})
axes[0, 1].set(title='Ordered paired outcomes', xlabel='Rank after sorting Delta', ylabel='Delta')

levels = np.array([80, 90, 95]); x = np.arange(3); width = 0.34
lml_coverage = [100 * E2_SUMMARY[f'lml_coverage_{level}'] for level in ('80', '90', '95')]
cv_coverage = [100 * E2_SUMMARY[f'cv_coverage_{level}'] for level in ('80', '90', '95')]
axes[1, 0].bar(x - width / 2, lml_coverage, width, label='LML', color='#59A14F')
axes[1, 0].bar(x + width / 2, cv_coverage, width, label='CV', color='#F28E2B')
axes[1, 0].plot(x, levels, 'ko--', label='Nominal')
axes[1, 0].set(xticks=x, xticklabels=[f'{v}%' for v in levels], ylim=(65, 100),
               title='Marginal interval coverage', xlabel='Nominal level', ylabel='Observed coverage (%)')
axes[1, 0].legend()

recovery_data, positions, labels, colors = [], [], [], []
for j, name in enumerate(PARAMETER_NAMES):
    for offset, method in ((-0.18, 'lml'), (0.18, 'cv')):
        recovery_data.append([r[f'{method}_log_parameter_error'][j] for r in e2_results])
        positions.append(j + 1 + offset); labels.append(method.upper()); colors.append('#59A14F' if method == 'lml' else '#F28E2B')
box = axes[1, 1].boxplot(recovery_data, positions=positions, widths=0.28, patch_artist=True, showfliers=True)
for patch, color in zip(box['boxes'], colors): patch.set_facecolor(color)
axes[1, 1].set(xticks=np.arange(1, 6), xticklabels=PARAMETER_NAMES,
               title='Absolute log-parameter recovery error', ylabel='abs(log(theta_hat/theta_true))')
axes[1, 1].legend([box['boxes'][0], box['boxes'][1]], ['LML', 'CV'])

fig.suptitle('E2: correctly specified locally periodic GP (50 frozen seeds)', fontsize=14)
fig.savefig(figure_dir / 'e2_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## 9. E3 — gradual period-drift misspecification (50 seeds)

Use the frozen phase transformation with `p(t)=10+2t/260` and integrated phase `phi(t)=130 log((10+2t/260)/10)`, while fitting the unchanged constant-period GP. **Warp only the periodic component:** retain the ordinary `(t-t')^2/(2 ell_d^2)` decay term and replace only the periodic phase difference by `phi(t)-phi(t')`. Do not pass `phi(t)` into the complete locally periodic kernel, because that would also warp the decay geometry. Reuse the E2 base parameters and indices and apply the symmetric H2 rule.

### E3 generator and covariance verification

The generator below changes only the periodic phase. Its RBF decay remains a function of the original time difference. The reference vector `(1, 50, 1, 11, 0.15)` records the unchanged base parameters and the midpoint period 11; because E3 has no single constant true period, period error is described as deviation from this nominal reference, not recovery of a true constant period.

In [ ]:
E3_THETA_REFERENCE = np.array([1.0, 50.0, 1.0, 11.0, 0.15])
E3_ENGINEERING_SEEDS = [999201]

def instantaneous_period(t):
    t = as_time_vector(t)
    return 10.0 + 2.0 * t / 260.0

def drift_phase(t):
    t = as_time_vector(t)
    return 130.0 * np.log((10.0 + 2.0 * t / 260.0) / 10.0)

def period_drift_latent_covariance(t1, t2, theta_reference=E3_THETA_REFERENCE):
    sigma_f, ell_d, ell_p, _, _ = np.asarray(theta_reference, dtype=float)
    t1, t2 = as_time_vector(t1), as_time_vector(t2)
    d = t1[:, None] - t2[None, :]
    phase_difference = drift_phase(t1)[:, None] - drift_phase(t2)[None, :]
    exponent = -(d ** 2) / (2.0 * ell_d ** 2) - 2.0 * np.sin(np.pi * phase_difference) ** 2 / ell_p ** 2
    return sigma_f ** 2 * np.exp(exponent)

def period_drift_training_covariance(t, theta_reference=E3_THETA_REFERENCE):
    t = as_time_vector(t)
    sigma_n = float(theta_reference[4])
    return period_drift_latent_covariance(t, t, theta_reference) + sigma_n ** 2 * np.eye(t.size)

def simulate_period_drift(seed):
    rng = np.random.default_rng(int(seed))
    t = np.arange(261, dtype=float)
    covariance = period_drift_training_covariance(t, E3_THETA_REFERENCE)
    return t, rng.multivariate_normal(np.zeros(t.size), covariance), E3_THETA_REFERENCE.copy()

e3_check_t = np.arange(261, dtype=float)
e3_check_covariance = period_drift_training_covariance(e3_check_t)
e3_symmetry_error = float(np.max(np.abs(e3_check_covariance - e3_check_covariance.T)))
e3_minimum_eigenvalue = float(np.linalg.eigvalsh(e3_check_covariance).min())
e3_phase_step = 1e-5
e3_probe = np.array([0.0, 65.0, 130.0, 195.0, 260.0])
e3_numeric_phase_derivative = (drift_phase(e3_probe + e3_phase_step) - drift_phase(e3_probe - e3_phase_step)) / (2.0 * e3_phase_step)
e3_phase_derivative_error = float(np.max(np.abs(e3_numeric_phase_derivative - 1.0 / instantaneous_period(e3_probe))))
assert np.isclose(instantaneous_period([0.0])[0], 10.0) and np.isclose(instantaneous_period([260.0])[0], 12.0)
assert e3_symmetry_error < 1e-12 and e3_minimum_eigenvalue > 0.0
assert e3_phase_derivative_error < 1e-8
E3_KERNEL_CHECK = {'period_at_0': 10.0, 'period_at_260': 12.0,
                   'symmetry_error': e3_symmetry_error,
                   'minimum_training_covariance_eigenvalue': e3_minimum_eigenvalue,
                   'maximum_phase_derivative_error': e3_phase_derivative_error}
pd.Series(E3_KERNEL_CHECK, name='E3 generator verification')

### Checkpointed E3 runner

Seed 999201 is a non-scientific engineering run used only to verify the new generator-to-checkpoint path. Scientific inference uses the complete frozen set `0,...,49`.

In [ ]:
def run_period_drift_seed(seed):
    t, y, theta_reference = simulate_period_drift(seed)
    train_idx, test_idx, folds = np.arange(250), np.arange(250, 261), inner_cv_blocks(250)
    started = time.perf_counter()
    lml_started = time.perf_counter()
    lml_fit = multistart_optimize(
        lambda z: negative_lml_and_grad(z, t[train_idx], y[train_idx]), analytic_gradient=True)
    lml_seconds = time.perf_counter() - lml_started
    cv_started = time.perf_counter()
    cv_fit = multistart_optimize(
        lambda z: blocked_cv_objective(z, t, y, folds, standardize_target=False),
        analytic_gradient=False)
    cv_seconds = time.perf_counter() - cv_started

    def evaluate_fit(fit):
        if not fit['success']:
            return None
        mean, covariance = gp_predict(t[train_idx], y[train_idx], t[test_idx],
                                      fit['theta'], observed=True)
        std = np.sqrt(np.maximum(np.diag(covariance), 0.0))
        metrics = {'jnlpd': joint_nlpd(y[test_idx], mean, covariance),
                   'rmse': float(np.sqrt(np.mean((y[test_idx] - mean) ** 2))),
                   'predictive_mean': mean.tolist(),
                   'predictive_covariance': covariance.tolist()}
        for level in (0.80, 0.90, 0.95):
            z_value = float(norm.ppf(0.5 + level / 2.0))
            lower, upper = mean - z_value * std, mean + z_value * std
            label = str(int(level * 100))
            metrics[f'coverage_{label}'] = float(np.mean((y[test_idx] >= lower) & (y[test_idx] <= upper)))
            metrics[f'mean_width_{label}'] = float(np.mean(upper - lower))
        return metrics

    lml_metrics, cv_metrics = evaluate_fit(lml_fit), evaluate_fit(cv_fit)
    lml_jnlpd = None if lml_metrics is None else lml_metrics['jnlpd']
    cv_jnlpd = None if cv_metrics is None else cv_metrics['jnlpd']
    result = {
        'complete': True, 'experiment': 'gradual_period_drift', 'seed': int(seed),
        'standardize_target': False, 'theta_reference': theta_reference.tolist(),
        'period_function': '10 + 2*t/260',
        'lml': _fit_checkpoint_summary(lml_fit), 'cv': _fit_checkpoint_summary(cv_fit),
        'test_targets': y[test_idx].tolist(), 'lml_metrics': lml_metrics, 'cv_metrics': cv_metrics,
        'lml_jnlpd': lml_jnlpd, 'cv_jnlpd': cv_jnlpd,
        'delta_cv_minus_lml': None if lml_jnlpd is None or cv_jnlpd is None else cv_jnlpd - lml_jnlpd,
        'lml_log_parameter_error': None if not lml_fit['success'] else absolute_log_parameter_error(lml_fit['theta'], theta_reference).tolist(),
        'cv_log_parameter_error': None if not cv_fit['success'] else absolute_log_parameter_error(cv_fit['theta'], theta_reference).tolist(),
        'lml_seconds': lml_seconds, 'cv_seconds': cv_seconds,
        'total_seconds': time.perf_counter() - started,
    }
    return _json_ready(result)

def run_checkpointed_period_drift(seeds, checkpoint_directory):
    return persistent_batch(seeds, checkpoint_directory, run_period_drift_seed,
                            'seed', 'seed', 'gradual_period_drift')

e3_engineering_results = run_checkpointed_period_drift(
    E3_ENGINEERING_SEEDS, Path('checkpoints') / 'engineering_e3')
pd.DataFrame([{'seed': r['seed'], 'lml_success': r['lml']['success'],
               'cv_success': r['cv']['success'], 'delta': r['delta_cv_minus_lml'],
               'lml_seconds': r['lml_seconds'], 'cv_seconds': r['cv_seconds']}
              for r in e3_engineering_results])

### Frozen scientific E3 run — seeds 0–49

All results are checkpointed after each seed. The scientific conclusion is made only from the complete frozen set; the run is not stopped or modified in response to interim outcomes.

In [ ]:
e3_results = run_checkpointed_period_drift(
    SCIENTIFIC_SEEDS, Path('checkpoints') / 'e3_period_drift')
assert len(e3_results) == 50 and {r['seed'] for r in e3_results} == set(SCIENTIFIC_SEEDS)
assert all(r['complete'] for r in e3_results)
E3_DELTAS, E3_SUMMARY, E3_REFERENCE_DEVIATION = summarize_e2(e3_results)
e3_summary_destination = Path('checkpoints') / 'e3_period_drift' / 'summary.json'
e3_summary_temporary = e3_summary_destination.with_suffix('.json.tmp')
e3_payload = {'summary': E3_SUMMARY, 'delta_records': E3_DELTAS.to_dict('records'),
              'reference_deviation': E3_REFERENCE_DEVIATION.to_dict('records'),
              'reference_note': 'No single constant period is true in E3; period error is deviation from nominal midpoint p=11, not true-parameter recovery.',
              'generator_check': E3_KERNEL_CHECK}
with e3_summary_temporary.open('w', encoding='utf-8') as handle:
    json.dump(_json_ready(e3_payload), handle, indent=2, sort_keys=True, allow_nan=False)
e3_summary_temporary.replace(e3_summary_destination)
display(pd.Series(E3_SUMMARY, name='E3 frozen summary'))
display(E3_DELTAS)
display(E3_REFERENCE_DEVIATION)
current_synthetic_report(E3_SUMMARY, 'E3')


### E3 visual evidence and frozen interpretation

The sign of `Delta = JNLPD(CV) - JNLPD(LML)` is interpreted exactly as frozen: negative values favour CV, positive values favour LML, and the paired-bootstrap interval determines whether H2 is supported, contradicted, or unresolved.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
e3_deltas = E3_DELTAS['delta_cv_minus_lml'].to_numpy()
axes[0, 0].hist(e3_deltas, bins=12, color='#4C78A8', edgecolor='white')
axes[0, 0].axvline(0, color='black', linestyle='--', linewidth=1)
axes[0, 0].axvline(E3_SUMMARY['mean_delta'], color='#E45756', linewidth=2, label='Mean')
axes[0, 0].set(title='E3 distribution of Delta', xlabel='JNLPD(CV) - JNLPD(LML)', ylabel='Seeds')
axes[0, 0].legend()

e3_ordered = np.sort(e3_deltas)
axes[0, 1].plot(np.arange(1, len(e3_ordered) + 1), e3_ordered, marker='o', markersize=3)
axes[0, 1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0, 1].text(0.03, 0.96,
                f"Mean Delta = {E3_SUMMARY['mean_delta']:.3f}\n"
                f"95% bootstrap CI for mean = [{E3_SUMMARY['bootstrap_95_low']:.3f}, {E3_SUMMARY['bootstrap_95_high']:.3f}]",
                transform=axes[0, 1].transAxes, va='top',
                bbox={'boxstyle': 'round', 'facecolor': '#F2CF5B', 'alpha': 0.3})
axes[0, 1].set(title='Ordered paired outcomes', xlabel='Rank after sorting Delta', ylabel='Delta')

levels = np.array([80, 90, 95]); x = np.arange(3); width = 0.34
e3_lml_coverage = [100 * E3_SUMMARY[f'lml_coverage_{level}'] for level in ('80', '90', '95')]
e3_cv_coverage = [100 * E3_SUMMARY[f'cv_coverage_{level}'] for level in ('80', '90', '95')]
axes[1, 0].bar(x - width / 2, e3_lml_coverage, width, label='LML', color='#59A14F')
axes[1, 0].bar(x + width / 2, e3_cv_coverage, width, label='CV', color='#F28E2B')
axes[1, 0].plot(x, levels, 'ko--', label='Nominal')
axes[1, 0].set(xticks=x, xticklabels=[f'{v}%' for v in levels], ylim=(50, 100),
               title='Marginal interval coverage', xlabel='Nominal level', ylabel='Observed coverage (%)')
axes[1, 0].legend()

e3_reference_data, e3_positions, e3_colors = [], [], []
for j, name in enumerate(PARAMETER_NAMES):
    for offset, method in ((-0.18, 'lml'), (0.18, 'cv')):
        e3_reference_data.append([r[f'{method}_log_parameter_error'][j] for r in e3_results])
        e3_positions.append(j + 1 + offset)
        e3_colors.append('#59A14F' if method == 'lml' else '#F28E2B')
e3_box = axes[1, 1].boxplot(e3_reference_data, positions=e3_positions, widths=0.28, patch_artist=True, showfliers=True)
for patch, color in zip(e3_box['boxes'], e3_colors): patch.set_facecolor(color)
axes[1, 1].set(xticks=np.arange(1, 6), xticklabels=PARAMETER_NAMES,
               title='Deviation from nominal reference', ylabel='abs(log(theta_hat/theta_reference))')
axes[1, 1].legend([e3_box['boxes'][0], e3_box['boxes'][1]], ['LML', 'CV'])
fig.suptitle('E3: gradual-period-drift generator, constant-period fitted GP (50 frozen seeds)', fontsize=14)
fig.savefig(figure_dir / 'e3_summary.png', dpi=180, bbox_inches='tight')
plt.show()

### E3 interpretation guide

Use the generated E3 report and tables above. An interval wholly below zero supports H2 for this controlled period-drift generator. Report remaining LML wins and any failed fits; a directional average does not imply universal superiority. Compare RMSE, marginal coverage and widths to distinguish improved means from wider uncertainty.

The generator's instantaneous period ranges approximately from 11.92 to 12.00 over the test block. Compare fitted periods and decay scales with that range as an empirical explanation, not causal proof. E3 reference-parameter errors measure deviation from the nominal parameters, because no single constant period is true in E3.

## 10. E4 — ten real sunspot outer blocks

Evaluate exactly ten consecutive 11-year blocks covering 1899–2008. For outer LML, compute mean/SD on the full outer-training history, optimise and predict in standardised space, then convert back. Inner CV uses a separate training-only mean/SD inside every fold; after CV selects theta, recompute mean/SD on the full outer-training history for the final outer prediction and convert back before scoring. Report original-unit JNLPD, RMSE, 80/90/95% marginal coverage, coverage error, interval width, wins, failures, and boundary hits. Use counts for ten-block wins and no significance test.

### E4 implementation and pre-outer engineering origin

The 1888–1898 block is outside the ten frozen outer blocks and is used only to verify the real-data pipeline. It cannot change the model, bounds, starts, folds, metrics, or final origins. Each inner CV fold calculates its own training-only standardisation; after selection, both methods use the full outer-training standardisation for the final forecast and are evaluated in original sunspot units.

In [ ]:
E4_ENGINEERING_ORIGINS = [1888]
E4_OUTER_ORIGINS = list(range(1899, 1999, 11))
assert E4_OUTER_ORIGINS == [1899, 1910, 1921, 1932, 1943, 1954, 1965, 1976, 1987, 1998]
assert E4_OUTER_ORIGINS[-1] + 10 == 2008

def predictive_metrics(targets, mean, covariance):
    targets = np.asarray(targets, dtype=float)
    mean, covariance = np.asarray(mean, dtype=float), np.asarray(covariance, dtype=float)
    std = np.sqrt(np.maximum(np.diag(covariance), 0.0))
    metrics = {'jnlpd': joint_nlpd(targets, mean, covariance),
               'rmse': float(np.sqrt(np.mean((targets - mean) ** 2))),
               'predictive_mean': mean.tolist(),
               'predictive_covariance': covariance.tolist()}
    for level in (0.80, 0.90, 0.95):
        z_value = float(norm.ppf(0.5 + level / 2.0))
        lower, upper = mean - z_value * std, mean + z_value * std
        label = str(int(level * 100))
        coverage = float(np.mean((targets >= lower) & (targets <= upper)))
        metrics[f'coverage_{label}'] = coverage
        metrics[f'coverage_error_{label}'] = abs(coverage - level)
        metrics[f'mean_width_{label}'] = float(np.mean(upper - lower))
        metrics[f'lower_{label}'] = lower.tolist()
        metrics[f'upper_{label}'] = upper.tolist()
    return metrics

def run_sunspot_outer_origin(test_start_year):
    test_start_year = int(test_start_year)
    years = sunspots['year'].to_numpy(dtype=float)
    targets = sunspots['sunspots'].to_numpy(dtype=float)
    origin_index = year_to_index(test_start_year)
    train_idx = np.arange(origin_index)
    test_idx = np.arange(origin_index, origin_index + 11)
    assert int(years[test_idx[0]]) == test_start_year and int(years[test_idx[-1]]) == test_start_year + 10
    assert train_idx[-1] + 1 == test_idx[0]
    train_mean, train_scale = fit_standardizer(targets[train_idx])
    z_train = standardize(targets[train_idx], train_mean, train_scale)
    folds = inner_cv_blocks(origin_index)
    started = time.perf_counter()

    lml_started = time.perf_counter()
    lml_fit = multistart_optimize(
        lambda z: negative_lml_and_grad(z, years[train_idx], z_train), analytic_gradient=True)
    lml_seconds = time.perf_counter() - lml_started

    cv_started = time.perf_counter()
    cv_fit = multistart_optimize(
        lambda z: blocked_cv_objective(z, years, targets, folds, standardize_target=True),
        analytic_gradient=False)
    cv_seconds = time.perf_counter() - cv_started

    def evaluate_gp(fit):
        if not fit['success']:
            return None
        mean_z, covariance_z = gp_predict(years[train_idx], z_train, years[test_idx],
                                          fit['theta'], observed=True)
        mean_y, covariance_y = unstandardize_prediction(
            mean_z, covariance_z, train_mean, train_scale)
        return predictive_metrics(targets[test_idx], mean_y, covariance_y)

    lml_metrics, cv_metrics = evaluate_gp(lml_fit), evaluate_gp(cv_fit)
    baseline_mean, baseline_covariance, baseline_variance = seasonal_naive_predictive(
        years[train_idx].astype(int), targets[train_idx], years[test_idx].astype(int))
    baseline_metrics = predictive_metrics(targets[test_idx], baseline_mean, baseline_covariance)
    lml_jnlpd = None if lml_metrics is None else lml_metrics['jnlpd']
    cv_jnlpd = None if cv_metrics is None else cv_metrics['jnlpd']
    return _json_ready({
        'complete': True, 'experiment': 'sunspot_outer_origin',
        'test_start_year': test_start_year,
        'train_year_range': [int(years[train_idx[0]]), int(years[train_idx[-1]])],
        'test_years': years[test_idx].astype(int).tolist(),
        'test_targets': targets[test_idx].tolist(),
        'outer_training_mean': train_mean, 'outer_training_scale': train_scale,
        'lml': _fit_checkpoint_summary(lml_fit), 'cv': _fit_checkpoint_summary(cv_fit),
        'lml_metrics': lml_metrics, 'cv_metrics': cv_metrics,
        'baseline_metrics': baseline_metrics, 'baseline_variance': baseline_variance,
        'lml_jnlpd': lml_jnlpd, 'cv_jnlpd': cv_jnlpd,
        'delta_cv_minus_lml': None if lml_jnlpd is None or cv_jnlpd is None else cv_jnlpd - lml_jnlpd,
        'lml_seconds': lml_seconds, 'cv_seconds': cv_seconds,
        'total_seconds': time.perf_counter() - started,
    })

def run_checkpointed_sunspot_origins(origins, checkpoint_directory):
    return persistent_batch(origins, checkpoint_directory, run_sunspot_outer_origin,
                            'test_start_year', 'origin', 'sunspot_outer_origin')

e4_engineering_results = run_checkpointed_sunspot_origins(
    E4_ENGINEERING_ORIGINS, Path('checkpoints') / 'engineering_e4')
display(pd.DataFrame([{'origin': r['test_start_year'],
                       'lml_success': r['lml']['success'], 'cv_success': r['cv']['success'],
                       'delta': r['delta_cv_minus_lml'],
                       'lml_seconds': r['lml_seconds'], 'cv_seconds': r['cv_seconds']}
                      for r in e4_engineering_results]))

### Frozen E4 outer evaluation — ten blocks

The ten blocks are evaluated once under the locked protocol. Comparisons are descriptive counts and paired block values; no significance test is applied to ten dependent historical blocks.

In [ ]:
e4_results = run_checkpointed_sunspot_origins(
    E4_OUTER_ORIGINS, Path('checkpoints') / 'e4_sunspots')
assert len(e4_results) == 10 and [r['test_start_year'] for r in e4_results] == E4_OUTER_ORIGINS
assert all(r['complete'] for r in e4_results)

e4_rows = []
for record in e4_results:
    for method in ('lml', 'cv', 'baseline'):
        metrics = record[f'{method}_metrics']
        row = {'test_start_year': record['test_start_year'], 'method': method.upper(),
               'jnlpd': metrics['jnlpd'], 'rmse': metrics['rmse']}
        for level in ('80', '90', '95'):
            row[f'coverage_{level}'] = metrics[f'coverage_{level}']
            row[f'coverage_error_{level}'] = metrics[f'coverage_error_{level}']
            row[f'mean_width_{level}'] = metrics[f'mean_width_{level}']
        e4_rows.append(row)
E4_METRICS = pd.DataFrame(e4_rows)
E4_DELTAS = pd.DataFrame({
    'test_start_year': [r['test_start_year'] for r in e4_results],
    'lml_jnlpd': [r['lml_jnlpd'] for r in e4_results],
    'cv_jnlpd': [r['cv_jnlpd'] for r in e4_results],
    'baseline_jnlpd': [r['baseline_metrics']['jnlpd'] for r in e4_results],
    'delta_cv_minus_lml': [r['delta_cv_minus_lml'] for r in e4_results],
})
E4_SUMMARY = {
    'origins': len(e4_results),
    'successful_pairs': int(sum(r['lml']['success'] and r['cv']['success'] for r in e4_results)),
    'mean_delta': float(E4_DELTAS['delta_cv_minus_lml'].mean()),
    'median_delta': float(E4_DELTAS['delta_cv_minus_lml'].median()),
    'cv_wins': int((E4_DELTAS['delta_cv_minus_lml'] < 0).sum()),
    'lml_wins': int((E4_DELTAS['delta_cv_minus_lml'] > 0).sum()),
    'exact_ties': int((E4_DELTAS['delta_cv_minus_lml'] == 0).sum()),
    'lml_failures': int(sum(not r['lml']['success'] for r in e4_results)),
    'cv_failures': int(sum(not r['cv']['success'] for r in e4_results)),
    'lml_continuations': int(sum(r['lml']['continuation_used'] for r in e4_results)),
    'cv_continuations': int(sum(r['cv']['continuation_used'] for r in e4_results)),
    'lml_boundary_hits': int(sum(sum(r['lml']['boundary_flags']) for r in e4_results if r['lml']['success'])),
    'cv_boundary_hits': int(sum(sum(r['cv']['boundary_flags']) for r in e4_results if r['cv']['success'])),
    'lml_beats_baseline': int((E4_DELTAS['lml_jnlpd'] < E4_DELTAS['baseline_jnlpd']).sum()),
    'cv_beats_baseline': int((E4_DELTAS['cv_jnlpd'] < E4_DELTAS['baseline_jnlpd']).sum()),
    'total_lml_seconds': float(sum(r['lml_seconds'] for r in e4_results)),
    'total_cv_seconds': float(sum(r['cv_seconds'] for r in e4_results)),
}
for method in ('LML', 'CV', 'BASELINE'):
    subset = E4_METRICS[E4_METRICS['method'] == method]
    E4_SUMMARY[f'{method.lower()}_mean_jnlpd'] = float(subset['jnlpd'].mean())
    E4_SUMMARY[f'{method.lower()}_median_jnlpd'] = float(subset['jnlpd'].median())
    E4_SUMMARY[f'{method.lower()}_mean_rmse'] = float(subset['rmse'].mean())
    for level in ('80', '90', '95'):
        E4_SUMMARY[f'{method.lower()}_coverage_{level}'] = float(subset[f'coverage_{level}'].mean())
        E4_SUMMARY[f'{method.lower()}_mean_width_{level}'] = float(subset[f'mean_width_{level}'].mean())

e4_summary_destination = Path('checkpoints') / 'e4_sunspots' / 'summary.json'
e4_summary_temporary = e4_summary_destination.with_suffix('.json.tmp')
with e4_summary_temporary.open('w', encoding='utf-8') as handle:
    json.dump(_json_ready({'summary': E4_SUMMARY,
                           'paired_blocks': E4_DELTAS.to_dict('records'),
                           'metrics': E4_METRICS.to_dict('records')}),
              handle, indent=2, sort_keys=True, allow_nan=False)
e4_summary_temporary.replace(e4_summary_destination)
display(pd.Series(E4_SUMMARY, name='E4 frozen summary'))
display(E4_DELTAS)
display(E4_METRICS)

### E4 visual evidence and frozen interpretation

With only ten dependent historical blocks, the complete paired values and win counts are reported descriptively. No bootstrap interval or significance test is attached to the real-data comparison.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
e4_origins = E4_DELTAS['test_start_year'].to_numpy()
for label, column, color in [('LML', 'lml_jnlpd', '#59A14F'), ('CV', 'cv_jnlpd', '#F28E2B'),
                             ('Seasonal naive', 'baseline_jnlpd', '#4C78A8')]:
    axes[0, 0].plot(e4_origins, E4_DELTAS[column], marker='o', label=label, color=color)
axes[0, 0].set(title='Joint predictive density by outer block', xlabel='Test-block start year',
               ylabel='JNLPD (lower is better)')
axes[0, 0].legend()

e4_delta_values = E4_DELTAS['delta_cv_minus_lml'].to_numpy()
e4_delta_colors = np.where(e4_delta_values < 0, '#F28E2B', '#59A14F')
axes[0, 1].bar(e4_origins.astype(str), e4_delta_values, color=e4_delta_colors)
axes[0, 1].axhline(0, color='black', linewidth=1)
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].set(title='Paired block differences', xlabel='Test-block start year',
               ylabel='JNLPD(CV) - JNLPD(LML)')

levels = np.array([80, 90, 95]); x = np.arange(3); width = 0.24
for offset, method, label, color in [(-width, 'lml', 'LML', '#59A14F'),
                                      (0, 'cv', 'CV', '#F28E2B'),
                                      (width, 'baseline', 'Seasonal naive', '#4C78A8')]:
    values = [100 * E4_SUMMARY[f'{method}_coverage_{level}'] for level in ('80', '90', '95')]
    axes[1, 0].bar(x + offset, values, width, label=label, color=color)
axes[1, 0].plot(x, levels, 'ko--', label='Nominal')
axes[1, 0].set(xticks=x, xticklabels=[f'{level}%' for level in levels], ylim=(60, 102),
               title='Marginal interval coverage', xlabel='Nominal level',
               ylabel='Observed coverage (%)')
axes[1, 0].legend()

for method, label, color in [('LML', 'LML', '#59A14F'), ('CV', 'CV', '#F28E2B'),
                             ('BASELINE', 'Seasonal naive', '#4C78A8')]:
    subset = E4_METRICS[E4_METRICS['method'] == method]
    axes[1, 1].plot(subset['test_start_year'], subset['rmse'], marker='o', label=label, color=color)
axes[1, 1].set(title='Point-forecast error by outer block', xlabel='Test-block start year',
               ylabel='RMSE (lower is better)')
axes[1, 1].legend()
fig.suptitle('E4: ten frozen 11-year sunspot forecast blocks', fontsize=14)
fig.savefig(figure_dir / 'e4_summary.png', dpi=180, bbox_inches='tight')
plt.show()

### E4 interpretation guide

The displayed tables are the current-run evidence for all ten blocks. Report the mean, median and strict-sign win counts together; do not hide a large outlying block. Compare joint density, RMSE and coverage separately, because the baseline can rank differently under these metrics. The baseline has diagonal predictive covariance, so these comparisons do not isolate the causal benefit of correlation modelling. Do not use significance tests on these ten dependent historical blocks.

## 11. Common preforecast activity groups

For each outer origin, calculate the 33rd and 67th percentiles of annual sunspot number using only that outer-training history. Assign each future year to low, medium, or high activity using the already observed value `y[t-11]`. The cutpoints and group assignment are therefore available before deployment and are identical for LML, CV, and the baseline. This is a **lag-11 activity proxy**, not an expectation from either GP. Small-group summaries are descriptive and their forecast indicators are not treated as independent.

## 12. E5 — calibration and support diagnostics

Reuse the frozen E4 predictions without refitting. Summarise marginal coverage and width over horizon bands 1–3, 4–7, and 8–11; repeat them over the common lag-11 activity groups; inspect standardized marginal residuals; and count negative Gaussian lower limits without clipping. These are descriptive diagnostics over dependent forecast points, not binomial significance tests. Negative limits demonstrate support mismatch with a non-negative target but do not by themselves identify the unique cause of forecast error.

In [ ]:
def horizon_band(horizon):
    if 1 <= horizon <= 3:
        return '1-3'
    if 4 <= horizon <= 7:
        return '4-7'
    if 8 <= horizon <= 11:
        return '8-11'
    raise ValueError('Horizon must be between 1 and 11')

def activity_proxy_group(value, lower_cut, upper_cut):
    if value <= lower_cut:
        return 'low'
    if value <= upper_cut:
        return 'medium'
    return 'high'

sunspot_lookup = dict(zip(sunspots['year'].astype(int), sunspots['sunspots'].astype(float)))
e5_rows, e5_cutpoint_rows = [], []
for record in e4_results:
    origin = int(record['test_start_year'])
    training_values = sunspots.loc[sunspots['year'] < origin, 'sunspots'].to_numpy(dtype=float)
    lower_cut, upper_cut = np.quantile(training_values, [1/3, 2/3])
    e5_cutpoint_rows.append({'test_start_year': origin, 'lower_cut': float(lower_cut),
                             'upper_cut': float(upper_cut), 'training_count': int(training_values.size)})
    for h, (year, target) in enumerate(zip(record['test_years'], record['test_targets']), start=1):
        proxy = float(sunspot_lookup[int(year) - 11])
        group = activity_proxy_group(proxy, lower_cut, upper_cut)
        for method in ('lml', 'cv', 'baseline'):
            metrics = record[f'{method}_metrics']
            mean = float(metrics['predictive_mean'][h - 1])
            variance = float(metrics['predictive_covariance'][h - 1][h - 1])
            std = float(np.sqrt(max(variance, 0.0)))
            row = {'test_start_year': origin, 'year': int(year), 'horizon': h,
                   'horizon_band': horizon_band(h), 'method': method.upper(),
                   'target': float(target), 'predictive_mean': mean,
                   'predictive_sd': std, 'error': float(target - mean),
                   'standardized_residual': float((target - mean) / std),
                   'lag11_activity_proxy': proxy, 'activity_group': group,
                   'activity_lower_cut': float(lower_cut), 'activity_upper_cut': float(upper_cut)}
            for level in ('80', '90', '95'):
                lower = float(metrics[f'lower_{level}'][h - 1])
                upper = float(metrics[f'upper_{level}'][h - 1])
                row[f'covered_{level}'] = bool(lower <= target <= upper)
                row[f'width_{level}'] = upper - lower
                row[f'negative_lower_{level}'] = bool(lower < 0.0)
            row['negative_mean'] = bool(mean < 0.0)
            e5_rows.append(row)
E5_POINTS = pd.DataFrame(e5_rows)
E5_CUTPOINTS = pd.DataFrame(e5_cutpoint_rows)
assert len(E5_POINTS) == 10 * 11 * 3
assert not E5_POINTS.isna().any().any()
assert (E5_POINTS.groupby(['test_start_year', 'year'])['activity_group'].nunique() == 1).all()

def calibration_slice(frame, grouping):
    rows = []
    group_columns = ['method', grouping]
    for keys, subset in frame.groupby(group_columns, sort=False):
        row = {'method': keys[0], grouping: keys[1], 'n': int(len(subset)),
               'rmse': float(np.sqrt(np.mean(subset['error'] ** 2))),
               'mean_error': float(subset['error'].mean()),
               'standardized_residual_mean': float(subset['standardized_residual'].mean()),
               'standardized_residual_sd': float(subset['standardized_residual'].std(ddof=0))}
        for level in ('80', '90', '95'):
            row[f'coverage_{level}'] = float(subset[f'covered_{level}'].mean())
            row[f'mean_width_{level}'] = float(subset[f'width_{level}'].mean())
            row[f'negative_lower_count_{level}'] = int(subset[f'negative_lower_{level}'].sum())
        rows.append(row)
    return pd.DataFrame(rows)

E5_HORIZON = calibration_slice(E5_POINTS, 'horizon_band')
E5_ACTIVITY = calibration_slice(E5_POINTS, 'activity_group')
e5_support_rows = []
for method, subset in E5_POINTS.groupby('method'):
    row = {'method': method, 'n': int(len(subset)),
           'negative_mean_count': int(subset['negative_mean'].sum()),
           'standardized_residual_mean': float(subset['standardized_residual'].mean()),
           'standardized_residual_sd': float(subset['standardized_residual'].std(ddof=0)),
           'abs_standardized_residual_gt_1_96': int((subset['standardized_residual'].abs() > 1.96).sum())}
    for level in ('80', '90', '95'):
        row[f'negative_lower_count_{level}'] = int(subset[f'negative_lower_{level}'].sum())
    e5_support_rows.append(row)
E5_SUPPORT = pd.DataFrame(e5_support_rows)

e5_directory = Path('checkpoints') / 'e5_diagnostics'; e5_directory.mkdir(parents=True, exist_ok=True)
e5_destination = e5_directory / 'summary.json'; e5_temporary = e5_destination.with_suffix('.json.tmp')
with e5_temporary.open('w', encoding='utf-8') as handle:
    json.dump(_json_ready({'horizon': E5_HORIZON.to_dict('records'),
                           'activity': E5_ACTIVITY.to_dict('records'),
                           'support': E5_SUPPORT.to_dict('records'),
                           'cutpoints': E5_CUTPOINTS.to_dict('records'),
                           'point_records': E5_POINTS.to_dict('records')}),
              handle, indent=2, sort_keys=True, allow_nan=False)
e5_temporary.replace(e5_destination)
display(E5_HORIZON)
display(E5_ACTIVITY)
display(E5_SUPPORT)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
method_colors = {'LML': '#59A14F', 'CV': '#F28E2B', 'BASELINE': '#4C78A8'}
band_order = ['1-3', '4-7', '8-11']; group_order = ['low', 'medium', 'high']
x = np.arange(3); width = 0.24
for offset, method in zip((-width, 0, width), ('LML', 'CV', 'BASELINE')):
    subset = E5_HORIZON[E5_HORIZON['method'] == method].set_index('horizon_band').loc[band_order]
    axes[0, 0].bar(x + offset, 100 * subset['coverage_90'], width, label=method, color=method_colors[method])
axes[0, 0].axhline(90, color='black', linestyle='--', linewidth=1, label='Nominal')
axes[0, 0].set(xticks=x, xticklabels=band_order, ylim=(45, 102), title='90% coverage by horizon band',
               xlabel='Forecast years', ylabel='Observed coverage (%)')
axes[0, 0].legend()

for offset, method in zip((-width, 0, width), ('LML', 'CV', 'BASELINE')):
    subset = E5_HORIZON[E5_HORIZON['method'] == method].set_index('horizon_band').loc[band_order]
    axes[0, 1].bar(x + offset, subset['mean_width_90'], width, label=method, color=method_colors[method])
axes[0, 1].set(xticks=x, xticklabels=band_order, title='90% interval width by horizon band',
               xlabel='Forecast years', ylabel='Mean interval width')
axes[0, 1].legend()

for offset, method in zip((-width, 0, width), ('LML', 'CV', 'BASELINE')):
    subset = E5_ACTIVITY[E5_ACTIVITY['method'] == method].set_index('activity_group').loc[group_order]
    axes[1, 0].bar(x + offset, 100 * subset['coverage_90'], width, label=method, color=method_colors[method])
axes[1, 0].axhline(90, color='black', linestyle='--', linewidth=1, label='Nominal')
axes[1, 0].set(xticks=x, xticklabels=group_order, ylim=(45, 102), title='90% coverage by lag-11 activity proxy',
               xlabel='Preforecast activity group', ylabel='Observed coverage (%)')
axes[1, 0].legend()

for offset, method in zip((-width, 0, width), ('LML', 'CV', 'BASELINE')):
    subset = E5_SUPPORT[E5_SUPPORT['method'] == method].iloc[0]
    counts = [subset[f'negative_lower_count_{level}'] for level in ('80', '90', '95')]
    axes[1, 1].bar(x + offset, counts, width, label=method, color=method_colors[method])
axes[1, 1].set(xticks=x, xticklabels=['80%', '90%', '95%'],
               title='Negative Gaussian lower limits', xlabel='Marginal interval level',
               ylabel='Count among 110 forecasts')
axes[1, 1].legend()
fig.suptitle('E5: calibration and Gaussian-support diagnostics', fontsize=14)
fig.savefig(figure_dir / 'e5_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

representative = next(r for r in e4_results if r['test_start_year'] == 1965)
rep_years = np.asarray(representative['test_years'])
rep_targets = np.asarray(representative['test_targets'])
fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
ax.plot(rep_years, rep_targets, 'ko-', linewidth=2, label='Observed')
for method, label, color in [('lml', 'LML', '#59A14F'), ('cv', 'CV', '#F28E2B'),
                             ('baseline', 'Seasonal naive', '#4C78A8')]:
    metrics = representative[f'{method}_metrics']
    mean = np.asarray(metrics['predictive_mean'])
    lower, upper = np.asarray(metrics['lower_90']), np.asarray(metrics['upper_90'])
    ax.plot(rep_years, mean, marker='o', color=color, label=label)
    ax.fill_between(rep_years, lower, upper, color=color, alpha=0.12)
ax.axhline(0, color='grey', linewidth=0.8)
ax.set(title='1965–1975 block: largest frozen LML–CV JNLPD divergence',
       xlabel='Year', ylabel='Annual sunspot number')
ax.legend(ncol=2)
fig.savefig(figure_dir / 'e4_1965_forecast.png', dpi=180, bbox_inches='tight')
plt.show()

### E5 interpretation guide

Use the current horizon, activity and support tables. Identify the worst coverage band from the output rather than assuming coverage decreases monotonically. Compare common lag-11 activity groups for both methods, and report negative prediction limits without clipping. These descriptive patterns can be consistent with mean bias, constant-noise assumptions and Gaussian support limitations, but do not identify a unique cause.

## 13. E6 — numerical failure and complexity

This constructed diagnostic deliberately leaves the fitted-model domain. Exact duplicate inputs and fixed noise values below the optimisation bound, including zero, expose the Cholesky failure mechanism. Only E6 varies numerical jitter; E1–E5 retain `j=1e-8`. The runtime benchmark measures covariance construction, Cholesky factorisation, and solves without claiming that a short finite benchmark must exhibit an exact cubic empirical slope. Exact dense GP algebra requires `O(n^3)` factorisation time and `O(n^2)` storage; one float64 `n` by `n` covariance alone occupies `8n^2` bytes.

In [ ]:
E6_JITTERS = np.array([0.0, 1e-14, 1e-12, 1e-10, 1e-8, 1e-6, 1e-4])
E6_NOISES = np.array([0.0, 1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2])
e6_base_t = np.linspace(0.0, 30.0, 61)
e6_t = np.concatenate([e6_base_t, [10.0, 20.0]])  # two exact duplicates
e6_y = np.sin(2.0 * np.pi * e6_t / 11.0) + 0.1 * np.cos(e6_t / 3.0)
e6_test_t = np.linspace(0.0, 35.0, 71)
e6_theta_zero_noise = np.array([1.0, 50.0, 1.0, 11.0, 0.0])

def conditioning_record(theta, jitter):
    covariance = training_covariance(e6_t, theta, jitter=jitter)
    condition_number = float(np.linalg.cond(covariance))
    minimum_eigenvalue = float(np.linalg.eigvalsh(covariance).min())
    try:
        cho_factor(covariance, lower=True, check_finite=True)
        success, message = True, 'success'
    except np.linalg.LinAlgError as exc:
        success, message = False, str(exc)
    return {'sigma_n': float(theta[4]), 'jitter': float(jitter),
            'condition_number': condition_number, 'minimum_eigenvalue': minimum_eigenvalue,
            'cholesky_success': success, 'message': message}

e6_jitter_records = [conditioning_record(e6_theta_zero_noise, jitter) for jitter in E6_JITTERS]
e6_noise_records = [conditioning_record(np.array([1.0, 50.0, 1.0, 11.0, noise]), 0.0)
                    for noise in E6_NOISES]
E6_JITTER_TABLE = pd.DataFrame(e6_jitter_records)
E6_NOISE_TABLE = pd.DataFrame(e6_noise_records)
assert not bool(E6_JITTER_TABLE.loc[E6_JITTER_TABLE['jitter'] == 0.0, 'cholesky_success'].iloc[0])
assert bool(E6_JITTER_TABLE.loc[E6_JITTER_TABLE['jitter'] == 1e-8, 'cholesky_success'].iloc[0])

reference_mean, reference_covariance = gp_predict(
    e6_t, e6_y, e6_test_t, e6_theta_zero_noise, observed=False, jitter=1e-8)
prediction_rows = []
for jitter in E6_JITTERS:
    try:
        mean, covariance = gp_predict(
            e6_t, e6_y, e6_test_t, e6_theta_zero_noise, observed=False, jitter=float(jitter))
        prediction_rows.append({'jitter': float(jitter), 'success': True,
                                'max_abs_mean_change_vs_1e-8': float(np.max(np.abs(mean - reference_mean))),
                                'max_abs_variance_change_vs_1e-8': float(np.max(np.abs(np.diag(covariance - reference_covariance))))})
    except np.linalg.LinAlgError:
        prediction_rows.append({'jitter': float(jitter), 'success': False,
                                'max_abs_mean_change_vs_1e-8': None,
                                'max_abs_variance_change_vs_1e-8': None})
E6_PREDICTION_STABILITY = pd.DataFrame(prediction_rows)

from threadpoolctl import threadpool_limits
E6_SIZES = np.array([200, 400, 800, 1600, 2400])
e6_runtime_rows = []
benchmark_theta = np.array([1.0, 50.0, 1.0, 11.0, 0.15])
for n in E6_SIZES:
    full_times, factor_times = [], []
    t_values = np.linspace(0.0, float(n - 1), int(n))
    y_values = np.sin(2.0 * np.pi * t_values / 11.0)
    repeats = 5 if n <= 800 else 3
    benchmark_covariance = training_covariance(t_values, benchmark_theta, jitter=JITTER)
    with threadpool_limits(limits=1):
        for _ in range(repeats):
            started = time.perf_counter()
            covariance = training_covariance(t_values, benchmark_theta, jitter=JITTER)
            factor = cho_factor(covariance, lower=True, check_finite=False)
            cho_solve(factor, y_values, check_finite=False)
            full_times.append(time.perf_counter() - started)
            started = time.perf_counter()
            factor = cho_factor(benchmark_covariance, lower=True, check_finite=False)
            cho_solve(factor, y_values, check_finite=False)
            factor_times.append(time.perf_counter() - started)
    e6_runtime_rows.append({'n': int(n), 'repeats': repeats,
                            'median_full_seconds': float(np.median(full_times)),
                            'median_factor_solve_seconds': float(np.median(factor_times)),
                            'matrix_bytes': int(8 * n ** 2),
                            'matrix_megabytes': float(8 * n ** 2 / 1024 ** 2)})
E6_RUNTIME = pd.DataFrame(e6_runtime_rows)
E6_EMPIRICAL_SLOPE_FULL = float(np.polyfit(np.log(E6_RUNTIME['n']), np.log(E6_RUNTIME['median_full_seconds']), 1)[0])
E6_EMPIRICAL_SLOPE_FACTOR = float(np.polyfit(np.log(E6_RUNTIME['n']), np.log(E6_RUNTIME['median_factor_solve_seconds']), 1)[0])

def runtime_log_fit(column):
    x = np.log(E6_RUNTIME['n'].to_numpy())
    y = np.log(E6_RUNTIME[column].to_numpy())
    slope, intercept = np.polyfit(x, y, 1)
    residual = y - (slope * x + intercept)
    return {'slope': float(slope), 'intercept': float(intercept),
            'r_squared': float(1 - np.sum(residual**2) / np.sum((y-y.mean())**2)),
            'n_values': E6_RUNTIME['n'].tolist(),
            'repeats': E6_RUNTIME['repeats'].tolist(),
            'fit': 'OLS log(median seconds) against log(n), all five sizes'}

E6_RUNTIME_FITS = {column: runtime_log_fit(column) for column in
                   ('median_full_seconds', 'median_factor_solve_seconds')}
display(pd.DataFrame(E6_RUNTIME_FITS).T)

e6_directory = Path('checkpoints') / 'e6_numerics'; e6_directory.mkdir(parents=True, exist_ok=True)
e6_destination = e6_directory / 'summary.json'; e6_temporary = e6_destination.with_suffix('.json.tmp')
E6_SUMMARY = {'empirical_slope_full_pipeline': E6_EMPIRICAL_SLOPE_FULL,
              'empirical_slope_factor_and_solve': E6_EMPIRICAL_SLOPE_FACTOR,
              'exact_gp_time_complexity': 'O(n^3)', 'exact_gp_storage_complexity': 'O(n^2)',
              'one_float64_covariance_bytes': '8*n^2', 'runtime_fits': E6_RUNTIME_FITS}
with e6_temporary.open('w', encoding='utf-8') as handle:
    json.dump(_json_ready({'summary': E6_SUMMARY,
                           'jitter': E6_JITTER_TABLE.to_dict('records'),
                           'noise': E6_NOISE_TABLE.to_dict('records'),
                           'prediction_stability': E6_PREDICTION_STABILITY.to_dict('records'),
                           'runtime': E6_RUNTIME.to_dict('records')}),
              handle, indent=2, sort_keys=True, allow_nan=False)
e6_temporary.replace(e6_destination)
display(E6_JITTER_TABLE)
display(E6_NOISE_TABLE)
display(E6_PREDICTION_STABILITY)
display(E6_RUNTIME)
display(pd.Series(E6_SUMMARY, name='E6 summary'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
successful_jitter = E6_JITTER_TABLE[E6_JITTER_TABLE['jitter'] > 0]
axes[0, 0].loglog(successful_jitter['jitter'], successful_jitter['condition_number'], 'o-')
axes[0, 0].axvline(JITTER, color='#E45756', linestyle='--', label='E1–E5 fixed jitter')
axes[0, 0].set(title='Duplicate-input conditioning', xlabel='Diagnostic jitter', ylabel='Condition number')
axes[0, 0].legend()

stable_predictions = E6_PREDICTION_STABILITY[E6_PREDICTION_STABILITY['success'] &
                                             (E6_PREDICTION_STABILITY['jitter'] > 0)]
axes[0, 1].loglog(stable_predictions['jitter'],
                  np.maximum(stable_predictions['max_abs_mean_change_vs_1e-8'], 1e-18),
                  'o-', label='Predictive mean')
axes[0, 1].loglog(stable_predictions['jitter'],
                  np.maximum(stable_predictions['max_abs_variance_change_vs_1e-8'], 1e-18),
                  's-', label='Predictive variance')
axes[0, 1].axvline(JITTER, color='#E45756', linestyle='--')
axes[0, 1].set(title='Prediction perturbation versus fixed-jitter reference',
               xlabel='Diagnostic jitter', ylabel='Maximum absolute change')
axes[0, 1].legend()

axes[1, 0].loglog(E6_RUNTIME['n'], E6_RUNTIME['median_full_seconds'], 'o-', label='Full GP algebra')
axes[1, 0].loglog(E6_RUNTIME['n'], E6_RUNTIME['median_factor_solve_seconds'], 's-', label='Factor + solve')
anchor = E6_RUNTIME['median_factor_solve_seconds'].iloc[-1] / E6_RUNTIME['n'].iloc[-1] ** 3
axes[1, 0].loglog(E6_RUNTIME['n'], anchor * E6_RUNTIME['n'] ** 3, '--', label='Cubic reference')
axes[1, 0].set(title='Exact dense GP computation', xlabel='Training size n', ylabel='Seconds')
axes[1, 0].legend()

axes[1, 1].plot(E6_RUNTIME['n'], E6_RUNTIME['matrix_megabytes'], 'o-')
axes[1, 1].set(title='Storage for one float64 covariance', xlabel='Training size n', ylabel='MiB = 8 n² / 2²⁰')
fig.suptitle('E6: numerical stability and exact-GP complexity', fontsize=14)
fig.savefig(figure_dir / 'e6_numerics_complexity.png', dpi=180, bbox_inches='tight')
plt.show()

### E6 interpretation guide

Read the current tables and figure above. Factorisation success alone does not establish numerical reliability: inspect conditioning and changes in predictions relative to jitter 1e-8. The constructed duplicate-input settings are diagnostics, not candidate fitted models.

Dense exact factorisation has O(n³) time and O(n²) storage. The finite-size measured slopes need not equal three. Report the current fit range, repetitions, R² and hardware environment; the full pipeline also includes quadratic covariance construction and does not include prediction. One float64 covariance matrix occupies 8n² bytes, which is only part of total working memory.

## 14. Results synthesis and limitations

The final evidence cell below generates the primary comparison from this run. Interpret the controlled E2/E3 contrast together with the heterogeneous real-data blocks and calibration diagnostics. Preserve contradictory outcomes; the conclusion must follow the completed frozen experiment, including failures.

### Findings from the completed persistent run

This synthesis was written after the frozen run completed; it changes no hypothesis, bound, split, model, seed or decision rule. The primary sign convention is Δ = JNLPD(CV) − JNLPD(LML), so a negative value favours CV.

Under correct specification (E2), all 50 pairs succeeded. Mean Δ was +0.076337 (median +0.023111), with a frozen 95% paired-bootstrap interval of [+0.027704, +0.137140]; CV won 17/50 seeds. Thus the directional mean evidence favours LML under this generator, supporting H1 in its frozen operational sense. Individual CV wins show that this is an average result, not a per-seed guarantee.

Under gradual period drift (E3), all 50 pairs succeeded. Mean Δ was −0.400703 (median −0.364298), interval [−0.500802, −0.301900], and CV won 44/50. The interval lies below zero, supporting H2 under this generator. Neither synthetic condition had an optimizer fit failure or continuation. The contrast is consistent with the idea that fitting training evidence and choosing parameters by future-like blocks can diverge when the fixed-period model is misspecified; it does not establish a universal misspecification theorem.

On the ten real sunspot blocks (E4), CV and LML each won five. Mean Δ was +0.157758 while median Δ was −0.026330. The 1965 block had Δ = +1.581063 and dominates the mean; omitting it as a descriptive influence check makes the remaining nine-block mean approximately −0.000387. This check is not a new winner rule. There were no failed fits, continuations or selected boundary hits. Mean JNLPD was 4.325 for LML, 4.482 for CV and 4.894 for the seasonal baseline, yet the baseline had the lowest mean RMSE (30.472 versus 31.718 and 33.234). The real-data comparison is therefore heterogeneous and sensitive to one block; it does not show a robust winner between the two GP selection rules.

E5 reveals forecast limitations. At the nominal 90% marginal level, overall coverage was 85.5% for LML, 78.2% for CV and 90.9% for the baseline. The middle horizons (4–7) had the poorest GP coverage (67.5% LML; 60.0% CV), while horizons 8–11 recovered to 100% and 92.5%; this does not support monotonic worsening with horizon. High lag-11 activity groups had lower GP coverage (70.2% LML; 63.8% CV) than low groups (100% and 93.8%), consistent with the activity part of H3. At the 95% level, Gaussian lower bounds were negative for 78/110 LML and 72/110 CV forecasts, demonstrating support mismatch with non-negative observations. The 110 forecast points come from ten dependent blocks, so these diagnostics are descriptive.

E6 confirms the dense exact-GP cost and sensitivity to numerical conditioning. Across n = 200, 400, 800, 1600 and 2400, the empirical log–log slope was 3.027 for factorisation plus solve (R² = 0.960) and 2.484 for the timed covariance-plus-factorisation pipeline (R² = 0.969). These finite-size timing fits are consistent with cubic algebra but are not measurements of a full hyperparameter optimisation.

Taken together, the controlled experiments answer the criterion question conditionally: LML had the lower mean future joint loss when the fitted family generated the data, while forecast-aligned CV had the lower mean under the frozen drifting-period generator. The historical sunspot series did not yield a stable method winner, and both GP variants showed calibration and support limitations. The result argues for checking deployment-shaped probabilistic scores even when LML is an efficient and coherent training criterion.

### Limitations and scope of inference

The synthetic evidence uses one kernel family, two generators, 50 seeds per condition, one sample length, four inner folds, and one 11-step forecast task. The real evidence contains only ten dependent historical blocks from one series, so this protocol treats win counts and slices descriptively and makes no significance claim. CV selection is based on only four validation blocks. A poor individual-block forecast can be compatible with selection variability or distribution change; these experiments do not isolate either cause. Although eight starts reduce local-optimum risk, they do not prove a global optimum.

The fitted model has zero mean after fold-specific standardisation, one constant period, constant Gaussian observation noise, and unconstrained Gaussian support. Predictive intervals condition on a point estimate of the hyperparameters and plug-in training mean/scale; uncertainty in those estimates is not integrated. This is an additional possible source of undercoverage, not a mechanism isolated by the experiments. The seasonal baseline deliberately uses diagonal covariance and is not an equally expressive joint model. JNLPD appropriately evaluates full predictive density but can be strongly affected by a badly placed or poorly conditioned joint distribution. The study therefore supports conditional claims about this frozen design, not general superiority across GP families, forecasting domains, or scoring rules.

Natural future refinements include a nonstationary phase model, activity-dependent noise or a non-negative likelihood, more historical validation origins, and forecast combinations. They are excluded here because adding them after observing outcomes would change the frozen research question rather than honestly answer it.

### Why tolerate the construction/deployment mismatch?

LML is not merely a training-error score: it integrates the latent function under the GP prior, balancing the quadratic fit term with a log-determinant term. It is a coherent, differentiable model-based selection criterion whose empirical merit is tested in E2. Nevertheless, its objective is the observed training sample, not the exact future block. Blocked CV aligns the score and temporal direction with deployment, at the cost of repeated fits and a limited set of historical validation blocks. Its alignment does not eliminate misspecification or guarantee good performance after a regime change.

Ideally, a forecast would match the unknown conditional distribution of all eleven future targets. That distribution is unavailable for the real series, so external JNLPD, marginal coverage, interval width and RMSE are complementary empirical checks. Strict propriety is an expected-score property of the logarithmic score over an appropriate class of distributions; it does not make a finite-sample Gaussian forecast calibrated or make the true distribution available within a restricted Gaussian family. E2/E3 offer controlled specification conditions, not a universal theorem about either selection rule.

Monitoring would retain the horizon/activity slices, boundary and convergence diagnostics, and original-unit joint scores used here. Potential remedies should target the observed limitation: a nonstationary phase model for changing recurrence, a suitable non-negative likelihood for support, or more validation origins for selection variability. These are future work, not post-result additions to this frozen study. E6's timed full pipeline is covariance construction plus factorisation and solve, not a complete gradient-based hyperparameter optimization or a forecast; its finite-size slopes must not be used to predict total experiment cost without counting objective evaluations.

## 15. Implementation log and reproducibility

### 15.1 Challenges and Solutions

This section records technical issues and their resolution independently of which tools helped identify them.

| Stage | Challenge and diagnosis | Resolution and verification | Limitation retained |
|---|---|---|---|
| Pre-E2 | The gradient formed an inverse-like matrix by solving against the identity, conflicting with the declared no-full-inverse policy. This was a policy/implementation mismatch, not evidence that the LML formula was mathematically wrong. | Evaluate each trace using `cho_solve(factor, D_j)` instead; E1 finite differences and the sklearn gradient comparison passed. | Matrix-right-hand-side solves still have cubic cost and matrix storage. |
| Pre-E2 | sklearn differentiates log variances, whereas this implementation differentiates log standard deviations. | Map amplitude/noise gradients by a factor of two, preserve the negative-LML sign, and verify against central differences. | Agreement at tested points is not a proof for every input. |
| Pre-E2 | Synthetic standardization and real-data fold preprocessing could be confused; real forecasts need original-unit scores. | Disable synthetic target scaling; compute real inner-fold scalers from training history, then refit using the outer-training scaler. E1 checks scaling, split indices and the JNLPD Jacobian identity. | No future data may determine a training transform. |
| Pre-E2 engineering | Different starts can find different optima; a lower returned objective need not be converged. | Freeze eight starts and the single recovery rule described in §5; keep start-level messages, boundaries and timings in atomic checkpoints. | Convergence does not certify global optimality. |
| E3 execution | The execution route initially used did not expose a working Jupyter workflow. | The launcher executed exact notebook-cell source and checkpointed seeds. Later the installed Jupyter environment was located and a fresh local kernel completed all 26 code cells. | The earlier workaround should not be described as proof that Jupyter was absent; cached replay is not a new model refit. |
| E6 launcher | The first helper omitted the `_json_ready` prerequisite. | Correct the helper's dependency list and rerun; saved E6 output was subsequently produced. | This was a launcher dependency error, not a GP correction. |
| E6 timing | Multithreaded finite-size timings gave noisy slope estimates. | Use a disclosed single-thread benchmark, separate covariance construction from factor-plus-solve, and record sizes, repeats, slopes and R². Preserve the original and fresh local timing summaries. | Benchmark revision occurred after inspecting timing quality; it is a disclosed engineering diagnostic, not a preregistered statistical finding. |
| Local replay | Jupyter's Windows connection-file permission setup raised `PermissionError` before any cell executed. | Rerun through the approved local launch with the required filesystem permissions; 26/26 cells completed, E1 passed 21/21, and there were zero cell errors. | This is local verification, not Colab or public-access verification. |

`review_evidence/failed_local_execution.ipynb` was the diagnostic snapshot from the last row: all code outputs were cleared and **zero code cells executed** before startup failed. It was moved to the Recycle Bin during the user-requested folder cleanup, along with superseded notebook copies; this log preserves the incident. The only current project notebook is `A2_GP_LML_vs_CV.ipynb` in the consolidated `review` folder.

### 15.2 Use of AI Tools

GenAI assisted with scope comparison, literature searching, derivation review, implementation scaffolding, test design, experiment runners, plotting, interpretation drafts and this documentation audit. These are material uses, not only proofreading. Assistance covered the explicit GP functions and reporting infrastructure; using the phrase “from scratch” describes the absence of an off-the-shelf GP in the scientific runs, not independent unaided authorship.

Suggestions were critically checked: the sklearn chain-rule mapping was verified; the earlier inverse-like construction was replaced to meet the numerical policy; synthetic scaling was rejected; non-converged solutions were not promoted to valid fits; and the E3 phase warp was applied only to the periodic term. E1 uses finite differences, sklearn fixed-parameter comparisons and SciPy log-density checks. Raw E3/E4 predictive distributions were also rescored independently using SciPy. A reported earlier cross-environment review is supplementary provenance; the directly archived executable evidence is the local notebook and checkpoint audit, not a second environment recreated in this update.

The H1 operationalization was recorded as a pre-E2 amendment, not backdated; its exact date remains in the central research-integrity record. The earlier local-run E3–E6 drafts were revised to retain contradictory findings: five E3 LML wins, the mixed real-data result, non-monotonic horizon coverage, and negative limits without clipping. The 1965 forecast illustration was selected after results because it had the largest paired divergence and is labelled post hoc. No scientific seed, model, split or optimizer setting was changed in response to this writing audit.

GenAI also assisted in separating this log, checking the actual specification/rubric, verifying GPML §5.4.3 equation (5.16), repairing stale prose and documenting release requirements. The author must review these explanations and take responsibility for the submitted claims; numerical agreement does not demonstrate the author's personal understanding.

### 15.3 Knowledge Gaps and Understanding Record

**Author confirmation is still required.** An assistant cannot truthfully declare that the author understands every component. The entries below identify specific understanding checks and evidence; they are not invented claims of completed study. Before submission, the author must mark what they can explain unaided and explicitly record any remaining gap, why the component is needed, how it was checked, and what they personally did to understand it.

| Component / why it is needed | Available correctness evidence | Understanding attempt to complete or confirm |
|---|---|---|
| Gaussian conditioning / full predictive distribution | sklearn mean and covariance agreement; E1 noise check | Derive the block-Gaussian posterior and explain latent versus observed covariance without reading code. |
| Five log-parameter gradients / efficient LML fitting | Central differences and mapped sklearn gradients | Derive the period term and factor-two amplitude/noise mapping; identify the corresponding code. |
| Cholesky / stable solves and log determinants | E1 density agreement; E6 duplicate-input diagnostics | Explain why successful factorization does not guarantee a well-conditioned problem and why jitter is not observation noise. |
| Blocked CV and scaling / prevent leakage and match deployment | E1 index/Jacobian checks; saved E4 predictions in original units | Work through one outer origin and its four training-only scalers, then explain final outer refitting. |
| SciPy L-BFGS-B / bounded numerical optimization | Logged termination messages, eight starts and recovery rule | Explain limited-memory curvature, bounds and line-search stopping at the level used here. The optimizer itself was not implemented from scratch. |

Unresolved items must remain disclosed rather than being changed to “understood” solely because tests pass.

### 15.4 Reproducibility and Release Status

The notebook records package versions, seeds, starts, bounds and optimizer options. `requirements-e1.txt` records the archived local environment; setup installs missing dependencies rather than forcing Colab to match it. The real series is the bundled statsmodels public-domain snapshot, not a fresh download of a revised modern solar series. No user-local source file is needed to generate the data.

The earlier local 26-cell execution and the first Colab run are historical records. The persistent Colab run completed on 23 September 2026 UTC with 27/27 code cells executed, no stored errors and 21/21 E1 checks. Its Drive bundle contains 50 E2 seeds, 50 E3 seeds and ten E4 origins, plus E5/E6 summaries, figures, a run manifest and a completion manifest. On 24 September, an independent read-only audit verified all 131 recorded SHA-256 checksums, the run fingerprint, every scientific record ID, the 10,000-resample bootstrap intervals, win counts and E4 paired differences against raw JSON. The final executed notebook retains those Colab outputs; its explanatory text was revised after execution without changing code or numerical outputs. This is disclosed rather than represented as a new end-to-end execution of the revised prose.

The shared Drive URL `https://drive.google.com/file/d/1ZluJ27VhK_v9QLT_Da2ZwbqxpMYJuFUd/view` opened in a signed-out browser on 24 September 2026 and displayed the executed Colab notebook. At that check, it contained an earlier version of six written sections. **Before submission, replace the shared file with this final merged notebook or issue a new public link, then verify the exact public/offline match from a signed-out browser.** Place the final URL as plain text in the single required journal PDF.

The final PDF must contain the report and this implementation log and follow the filename `STUDENTNAME_STUDENTID_YEAR_UTS_ML_Journal.pdf`. The author's personal knowledge-gap review, PDF assembly, final public-version check and visual review remain open. Passing numerical checks does not substitute for the author's Q&A understanding.

**Recovery and writing revisions.** The first Colab run lost temporary checkpoint files when its runtime ended. The persistent rerun used mounted Google Drive, validated checkpoint provenance and completed its evidence bundle. AI assisted the recovery implementation, source checking and post-run synthesis. The scientific code and frozen choices were not changed after the outcomes were seen.


## 16. References

- Williams, C. K. I., & Rasmussen, C. E. Gaussian Processes for Regression. *Advances in Neural Information Processing Systems 8*, 514–520 (NIPS 1995 conference; proceedings published 1996). §§2–3: conditional prediction, relation to Kriging, hyperparameter optimization/integration. https://proceedings.neurips.cc/paper_files/paper/1995/file/7cce53cf90577442771720a370c3c723-Paper.pdf
- Rasmussen, C. E., & Williams, C. K. I. (2006). *Gaussian Processes for Machine Learning*. MIT Press. Chapter 2: posterior conditioning and marginal likelihood; chapter 4: covariance functions; chapter 5 §§5.2–5.4: selection, including equation (5.16), p. 120. https://gaussianprocess.org/gpml/chapters/RW2.pdf ; https://gaussianprocess.org/gpml/chapters/RW4.pdf ; https://gaussianprocess.org/gpml/chapters/RW5.pdf
- Bachoc, F. (2013). Cross validation and maximum likelihood estimations of hyper-parameters of Gaussian processes with model misspecification. *Computational Statistics & Data Analysis*, 66, 55–69. §§2–5: variance-based risk, specification and numerical comparisons. https://doi.org/10.1016/j.csda.2013.03.016 ; author manuscript: https://arxiv.org/pdf/1301.4320
- Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting*, 16(4), 437–450. Rolling origins, recalibration and multiple test periods. https://doi.org/10.1016/S0169-2070(00)00065-0
- Gneiting, T., & Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *Journal of the American Statistical Association*, 102(477), 359–378. §4.1, equation (19): density log score and KL divergence. https://doi.org/10.1198/016214506000001437 ; https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jasa.pdf
- WDC-SILSO, Royal Observatory of Belgium. *Table of minima, maxima and cycle durations*. Duration context based on smoothed monthly Version 2.0 data. https://www.sidc.be/SILSO/cyclesmm
- European Space Agency (2007, December 4). *SOHO — keeping an eye on the Sun for 12 years*. https://www.esa.int/Science_Exploration/Space_Science/SOHO_-_keeping_an_eye_on_the_Sun_for_12_years
- statsmodels developers. *Yearly sunspots data 1700–2008*. Dataset definition, NGDC source and public-domain notice. https://www.statsmodels.org/stable/datasets/generated/sunspots.html
- Valinia, A., et al. (2022). *Safe Human Expeditions Beyond Low Earth Orbit (LEO)*. NASA/TM-20220002905, §8.3.3. Historical corroborating source cited in the freeze record; primary numerical duration support here is SILSO. https://ntrs.nasa.gov/citations/20220002905

Web source checks for this writing revision: 23 September 2026. The NASA PDF could not be retrieved during this audit; no new exact quotation or page claim is based on it.

## 17. Completion gate and persistent evidence bundle

Run this cell last. It validates the saved records and creates the evidence ZIP in Google Drive.

In [ ]:

# Final gate: numerical evidence is persistent; notebook saving is a separate step.
expected = {
    'e2_well_specified': ('seed', 'seed', list(range(50))),
    'e3_period_drift': ('seed', 'seed', list(range(50))),
    'e4_sunspots': ('origin', 'test_start_year', list(range(1899,1999,11))),
}
for folder, (prefix, key, identifiers) in expected.items():
    for identifier in identifiers:
        file = RUN_ROOT / 'checkpoints' / folder / f'{prefix}_{identifier}.json'
        record = json.loads(file.read_text(encoding='utf-8'))
        assert record['complete'] and record[key] == identifier
        assert record['run_fingerprint'] == RUN_FINGERPRINT
for folder in (*expected, 'e5_diagnostics', 'e6_numerics'):
    json.loads((RUN_ROOT/'checkpoints'/folder/'summary.json').read_text(encoding='utf-8'))
assert len(E1_RESULTS) == 21 and all(row['passed'] for row in E1_RESULTS)

comparison = pd.DataFrame([
    {'experiment': label, 'mean_delta': s['mean_delta'], 'median_delta': s['median_delta'],
     'CV_wins': s['cv_wins'], 'pairs': s['successful_pairs']}
    for label,s in [('E2',E2_SUMMARY),('E3',E3_SUMMARY),('E4',E4_SUMMARY)]])
display(comparison)
comparison.to_csv(RUN_ROOT/'current_run_comparison.csv',index=False)
report = '# Current-run synthesis\n\n'
for label,s in [('E2',E2_SUMMARY),('E3',E3_SUMMARY)]:
    report += (f"{label}: mean Δ {s['mean_delta']:+.6f}, median {s['median_delta']:+.6f}, "
               f"95% interval [{s['bootstrap_95_low']:+.6f}, {s['bootstrap_95_high']:+.6f}], "
               f"CV wins {s['cv_wins']}/{s['successful_pairs']}.\n\n")
report += (f"E4: mean Δ {E4_SUMMARY['mean_delta']:+.6f}, median "
           f"{E4_SUMMARY['median_delta']:+.6f}; CV wins {E4_SUMMARY['cv_wins']}/10. "
           'This is descriptive evidence from dependent historical blocks.\n\n')
report += 'Interpret this table alongside the complete distributions, failures, calibration and baseline comparisons.\n'
(RUN_ROOT/'current_run_synthesis.md').write_text(report,encoding='utf-8')
display(Markdown(report))

evidence = [p for p in RUN_ROOT.rglob('*') if p.is_file()
            and p.suffix in {'.json','.png','.md','.csv'}
            and p.name != 'completion_manifest.json']
hashes = {str(p.relative_to(RUN_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in evidence}
atomic_json(RUN_ROOT/'completion_manifest.json', {
    'completed_utc': datetime.now(timezone.utc).isoformat(), 'run_fingerprint': RUN_FINGERPRINT,
    'e1_passes':21, 'scientific_records':110, 'sha256':hashes,
    'note':'Evidence complete. Save/download the executed notebook separately; browser output completeness is not asserted.'})
bundle = RUN_ROOT/'A2_evidence.zip'
temporary = RUN_ROOT/'A2_evidence.zip.tmp'
with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED) as z:
    for p in evidence+[RUN_ROOT/'completion_manifest.json']:
        z.write(p,str(p.relative_to(RUN_ROOT)))
with zipfile.ZipFile(temporary) as z:
    assert z.testzip() is None
temporary.replace(bundle)
print('COMPLETE: 21 E1 checks; 50 E2 + 50 E3 + 10 E4 records; E5/E6 summaries.',flush=True)
print('Evidence ZIP saved to:',bundle,flush=True)
print('Now save this notebook in Colab and download its executed .ipynb.',flush=True)
print('If a display is missing, rerun its cell with the same mounted Drive run.',flush=True)
